## Criação da base de dados principal com informações das estações de monitoramento no Brasil (MQAr_BR)

##### Bibliotecas

In [2]:
import os, time, math, requests, pandas as pd
from datetime import datetime, timedelta, timezone
import pandas as pd
from collections import defaultdict
import re
import numpy as np
from pathlib import Path
import difflib
import unicodedata
from typing import Callable, Any, Optional
from functools import partial
from pandas.api.types import is_object_dtype, is_string_dtype
from collections import Counter
import unicodedata as ud

In [3]:
import scripts.createUfStations_functions as cuf
import scripts

##### Dicionários e listas 

In [4]:
# Colunas que precisam conter nas files de dados de ESTAÇÃO para cada estado
mqar_campos = [
        "UF","ID_OEMA","CIDADE","ID_MMA","ID_MMA_COMPLETO","POLUENTE","COD_POLUENTE",
        "CD_MUN","COD_UF_IBGE","PROPRIETARIO","PROP_ENTIDADE","OPERADOR","OP_ENTIDADE",
        "LATITUDE","LONGITUDE","MOBILIDADE","CATEGORIA","FUNCIONAMENTO","METODO",
        "MARCA",'INICIO', 'FIM',"FINALIDADE","MONITORAR","FONTE","CALIBRACAO","REALOCACAO",
        "OBS_CALIBRACAO","DADOS_MONITORAMENTO","RECONHECIDA","OBS_GERAIS",
        "STATUS","CERTIFICACAO","REP_ESPACIAL_DECLARADA"
    ]

In [5]:
name_to_uf = {
    "acre":"AC","alagoas":"AL","amapa":"AP","amazonas":"AM","bahia":"BA","ceara":"CE",
    "distrito federal":"DF","espirito santo":"ES","goias":"GO","maranhao":"MA",
    "mato grosso":"MT","mato grosso do sul":"MS","minas gerais":"MG","para":"PA",
    "paraiba":"PB","parana":"PR","pernambuco":"PE","piaui":"PI","rio de janeiro":"RJ",
    "rio grande do norte":"RN","rio grande do sul":"RS","rondonia":"RO","roraima":"RR",
    "santa catarina":"SC","sao paulo":"SP","sergipe":"SE","tocantins":"TO"
}

In [6]:
UF_TO_IBGE = {
    "AC":12,"AL":27,"AP":16,"AM":13,"BA":29,"CE":23,"DF":53,"ES":32,"GO":52,"MA":21,
    "MT":51,"MS":50,"MG":31,"PA":15,"PB":25,"PR":41,"PE":26,"PI":22,"RJ":33,"RN":24,
    "RS":43,"RO":11,"RR":14,"SC":42,"SP":35,"SE":28,"TO":17
}

def sigla_to_ibge(uf): return UF_TO_IBGE[uf.upper()]

In [7]:
# Importar planilha com os códigos de poluentes
base = Path.cwd().parent  
out_dir = base / "data" / "dicionarios" 
out_dir.mkdir(parents=True, exist_ok=True)

df_cod = pd.read_csv(out_dir / 'CODIGO_POLUENTES.csv')

In [8]:
# Importar planilha com as respostas do formulário das UFs
base = Path.cwd().parent  
fr_dir = base / "data" 
fr_dir.mkdir(parents=True, exist_ok=True)

forms = pd.read_csv(fr_dir / '2025_Formulário_Coleta_Respostas_UFs.csv')

#Indice das colunas com respostas sobre rede de monitoramento
# for i, c in enumerate(forms.columns):
#    print(f"[{i}] {c}")

idxs = [6,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22] 

#### Passo a passo para criação da planilha MQAr completa:

__Sequência de ações para criar planilha final completa:__

1. Fazer upload de todas as planilhas UF_estacoes e conferir a quantidade de estações por planilha
2. Limpar e padronizar caracteres
3. Unir DFs e substituir categorias com escrita errada e padronizar (ex. FUNCIONAMENTO - Sim = Ativa)
4. Conferir nomes de poluentes e substituir pelo dicionário quando diferente (ex. PM10 = MP10)
5. Conferir linhas repetidas ou informações diferentes
6. Comparar planilha final com planilha PurpleAir e MQAr do ano anterior
7. Criar ID_MMA_COMPLETO
8. Explodir poluentes - um poluente por linha
9. Salvar e exportar

1. Fazer upload de todas as planilhas UF_estacoes e conferir a quantidade de estações por planilha

In [781]:
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_ESTACOES" 
df_dir.mkdir(parents=True, exist_ok=True)
ufs_dfs = cuf.load_csvs(df_dir, prefix=None, recursive=False, limit=None)

Loaded AC_estacoes.csv: (30, 34)
Loaded AL_estacoes.csv: (42, 34)
Loaded BA_estacoes.csv: (15, 31)
Loaded CE_estacoes.csv: (4, 34)
Loaded DF_estacoes.csv: (9, 31)
Loaded ES_estacoes.csv: (19, 31)
Loaded MA_estacoes.csv: (14, 31)
Loaded MG_estacoes.csv: (68, 28)
Loaded MS_estacoes.csv: (4, 34)
Loaded MT_estacoes.csv: (5, 31)
Loaded PB_estacoes.csv: (4, 34)
Loaded PE_estacoes.csv: (4, 34)
Loaded PR_estacoes.csv: (28, 31)
Loaded RJ_estacoes.csv: (100, 35)
Loaded RR_estacoes.csv: (3, 34)
Loaded RS_estacoes.csv: (21, 31)
Loaded SC_estacoes.csv: (4, 31)
Loaded SP_estacoes.csv: (102, 31)


In [782]:
for name, d in ufs_dfs.items():
    print(f"\n=== {name} ===")
    display(d)


=== AC_estacoes ===


,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,AC,AcreBioClima - UFAC,Rio Branco,NaN,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,2023.0,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
1,AC,Ministério Público do Estado do Acre (SEDE),Rio Branco,NaN,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,2023.0,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
2,AC,MPAC_ABR_01_promotoria,Assis Brasil,NaN,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,2023.0,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
3,AC,MPAC_ABR_02_SEMSA,Assis Brasil,NaN,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,2023.0,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
4,AC,MPAC_ACL_01_promotoria,Acrelandia,NaN,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,2023.0,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
5,AC,MPAC_BJR_01_promotoria,Bujari,NaN,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,2023.0,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
6,AC,MPAC_BRL_01_promotoria,Brasiléia,NaN,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,2023.0,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
7,AC,MPAC_BRL_02_radio fm 90.3,Brasiléia,NaN,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,2023.0,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
8,AC,MPAC_CPX_01_qpm,Capixaba,NaN,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,2023.0,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
9,AC,MPAC_CZS_02_ciosp,Cruzeiro do Sul,NaN,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,2023.0,2023.0,NaN,NaN,NaN,NaN,NaN,NaN



=== AL_estacoes ===


,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,AL,Frigorifico caprisu,Santa Luzia do Norte,NaN,NaN,"PTS,MP10,MP25,SO2,NO2,O3,CO",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AL,Braskem,Maceio,NaN,NaN,"PTS,MP10,MP25,SO2,NO2,NO,NOX,O3,CO",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AL,QAR01-PAU FERRO,Craibas,NaN,NaN,"PTS,MP10,MP25,SO2",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AL,QAR02-LAGOA DA CRUZ,Craibas,NaN,NaN,"PTS,MP10,MP25,SO2",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AL,QAR03-LAGOA DO MEL,Craibas,NaN,NaN,"PTS,MP10,MP25,SO2",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,AL,QAR 04 - PLANTA DE BENEFICIAMENTO,Craibas,NaN,NaN,"PTS,MP10,MP25,SO2",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,AL,[A303] MACEIÓ,Maceio,NaN,NaN,"PTS,MP10,MP25,03,CO,SO2,NO2,FMC",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,AL,Fabrica frango favorito,Santa Luzia do Norte,NaN,NaN,"PTS,MP10,MP25,SO2,NO2,CO,O3",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== BA_estacoes ===


...

,UF,COD_UF_IBGE,ID_OEMA,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,PROP_ENTIDADE,...,FIM,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA
0,RS,43.0,Sapucaia,RS0001,Sapucaia do Sul,4320008.0,Referencia,Automatica,FEPAM,Pública,...,2010-01-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala urbana
1,RS,43.0,Canoas VCOMAR,RS0004,Canoas,4304606.0,Referencia,Automatica,FEPAM,Pública,...,2009-12-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala de Bairro
2,RS,43.0,Canoas P Universitário,RS0005,Canoas,4304606.0,Referencia,Automatica,Refap,Pública,...,-,Ativa,A cada 1 mês,NaN,Sim,Consulta 2025,NaN,Sim,Sim,Escala de Bairro
3,RS,43.0,Esteio Vila Ezequiel,RS0006,Canoas,4304606.0,Referencia,Automatica,Refap,Pública,...,2019-12-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala urbana
4,RS,43.0,Triunfo DEPREC,RS0009,Triunfo,4322004.0,Referencia,Automatica,Tractebel,Privada,...,2018-12-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala urbana
5,RS,43.0,Triunfo DEPREC,RS0009,Triunfo,4322004.0,Referencia,Automatica,Tractebel,Privada,...,2018-12-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala urbana
6,RS,43.0,Triunfo DEPREC,RS0009,Triunfo,4322004.0,Referencia,Automatica,Tractebel,Privada,...,2018-12-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala urbana
7,RS,43.0,Gravataí C Jardim Timbaúva,RS0012,Gravataí,4309209.0,Referencia,Automatica,FEPAM,Pública,...,-,Ativa,A cada 2 meses,Checagem mensal,Sim,Consulta 2025,NaN,Sim,Sim,Escala urbana
8,RS,43.0,Charqueadas-AT,RS0013,Charqueadas,4305355.0,Referencia,Automatica,Tractebel,Privada,...,2018-12-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala urbana
9,RS,43.0,Guaiba Parque 35,RS0015,Guaíba,4309308.0,Referencia,Automatica,CMPC,Privada,...,-,Ativa,A cada 1 mês,NaN,Sim,Consulta 2025,NaN,Sim,Sim,Escala urbana



=== SC_estacoes ===


,UF,COD_UF_IBGE,ID_OEMA,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,PROP_ENTIDADE,...,FIM,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA
0,SC,42,Vila Moema,SC0001,Tubarão,4218707,Referencia,Automatica,Diamante Geração De EnergiaLTDA.,Privada,...,NaN,Ativa,NaN,NaN,Sim,Consulta 2025,NaN,NaN,NaN,NaN
1,SC,42,Capivari,SC0002,Capivari de Baixo,4203956,Referencia,Automatica,Diamante Geração De EnergiaLTDA.,Privada,...,NaN,Ativa,NaN,NaN,Sim,Consulta 2025,NaN,NaN,NaN,NaN
2,SC,42,São Bernardo,SC0003,Tubarão,4218707,Referencia,Automatica,Diamante Geração De EnergiaLTDA.,Privada,...,NaN,Ativa,NaN,NaN,Sim,Consulta 2025,NaN,NaN,NaN,NaN
3,SC,42,UFSC,SC0004,Florianopolis,4205407,Referencia,Automatica,IMA,Publica,...,NaN,Ativa,"Mensal(O3),Mensal(NO2),Mensal(MP25)",NaN,Não,Consulta 2025,NaN,NaN,NaN,NaN



=== SP_estacoes ===


,ID_OEMA,CALIBRACAO,CATEGORIA,CD_MUN,CIDADE,COD_UF_IBGE,FIM,FINALIDADE,FONTE,FUNCIONAMENTO,...,OP_ENTIDADE,POLUENTE,PROPRIETARIO,PROP_ENTIDADE,REALOCACAO,REP_ESPACIAL,REP_ESPACIAL_DECLARADA,STATUS,UF,RECONHECIDA
0,Americana,diária,Referencia,3501608.0,Americana,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,O3,ERT",CETESB,Pública,Não,NaN,Bairro,Ativa,SP,NaN
1,Americana-Vila Sta Maria,NaN,Referencia,3501608.0,Americana,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,O3,ERT",CETESB,Pública,NaN,NaN,NaN,Inativa,SP,NaN
2,Araraquara,diária,Referencia,3503208.0,Araraquara,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,NO,O3,NO2,NOX",CETESB,Pública,Não,NaN,Bairro,Ativa,SP,NaN
3,Araçatuba,diária,Referencia,3502804.0,Araçatuba,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,NO,O3,NO2,NOX",CETESB,Pública,Não,NaN,Urbana,Ativa,SP,NaN
4,Bauru,diária,Referencia,3506003.0,Bauru,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,NO,O3,NO2,NOX,MP25",CETESB,Pública,Não,NaN,Bairro,Ativa,SP,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,Taboão da Serra,diária,Referencia,3552809.0,Taboão da Serra,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,NO,CO,NO2,NOX,MP25",CETESB,Pública,Não,NaN,Micro,Ativa,SP,NaN
98,Tatuapé,NaN,Referencia,3550308.0,São Paulo,35.0,NaN,Fornecer dados,Coleta Interna,Manual,...,Pública,FMC,CETESB,Pública,Não,NaN,Média,Sim,SP,Nao
99,Tatuí,diária,Referencia,3554003.0,Tatuí,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,NO,O3,NO2,NOX",CETESB,Pública,Não,NaN,Urbana,Ativa,SP,NaN
100,Taubaté,diária,Referencia,3554102.0,Taubaté,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,NO,CO,O3,SO2,NO2,NOX,MP25",CETESB,Pública,Não,NaN,Bairro,Ativa,SP,NaN


#### ID_MMA intermediário

In [784]:
def _ascii_lower(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = ud.normalize("NFKD", s)
    s = "".join(ch for ch in s if not ud.combining(ch))
    return s.lower()


def _normalize_dt_mixed(series: pd.Series, anchor="start") -> pd.Series:
    s = pd.to_datetime(series, errors="coerce", utc=True)
    raw = series.astype("string").str.strip().str.replace(r"[\/\.]", "-", regex=True)

    # YYYY-MM
    mask_ym = raw.str.match(r"^\d{4}-\d{1,2}$", na=False)
    if mask_ym.any():
        base = pd.to_datetime(raw[mask_ym] + "-01", errors="coerce", utc=True)
        s.loc[mask_ym] = base if anchor == "start" else base + pd.offsets.MonthEnd(0)

    # YYYY
    mask_y = raw.str.match(r"^\d{4}$", na=False)
    if mask_y.any():
        s.loc[mask_y] = pd.to_datetime(
            raw[mask_y] + ("-01-01" if anchor == "start" else "-12-31"),
            errors="coerce",
            utc=True,
        )

    return s.dt.tz_convert(None)


def _parse_existing_nums(id_series: pd.Series, uf: str) -> pd.Series:
    """Extrai os 4 dígitos finais dos IDs válidos daquela UF."""
    pat = f"^{uf}\\d{{4}}$"
    m = id_series.astype("string").str.fullmatch(pat, na=False)
    nums = id_series.where(m).str[-4:].astype("Int64", errors="ignore")
    return nums


def assign_id_mma_all_rules(
    df: pd.DataFrame,
    uf_col="UF",
    start_col="INICIO",
    station_col="ID_OEMA",
    id_col="ID_MMA",
    anchor="start",
    pr_fixed_order=None,  # lista em ordem das estações PR com IDs fixos
    blocked_ufs=("RJ", "ES", "SC", "PE", "PB", "MA", "CE", "BA"),  # não criar nem alterar
):
    """
    Regras:
    1) Em cada UF, ordenar por INICIO asc. Empate ou sem data por nome A>Z.
    2) SP: manter existentes e preencher só vazios, continuando após o maior existente.
    3) DF: mesmo de SP.
    4) PR: manter existentes e os da lista fixa; criar novos após o maior existente.
    5) UFs em blocked_ufs: não criar nem alterar.
    6) Outras UFs: preencher só vazios, iniciando de 0001 ou após o maior existente.
    """
    out = df.copy()

    for c in [uf_col, station_col]:
        out[c] = out.get(c, pd.Series(pd.NA, index=out.index)).astype("string")
    if id_col not in out.columns:
        out[id_col] = pd.Series(pd.NA, index=out.index, dtype="string")
    else:
        out[id_col] = out[id_col].astype("string")

    out[start_col] = _normalize_dt_mixed(out[start_col], anchor=anchor)
    name_key = out[station_col].map(_ascii_lower)
    start_key = out[start_col].fillna(pd.Timestamp.max)
    out = (
        out.assign(_name_key=name_key, _start_key=start_key)
        .sort_values([uf_col, "_start_key", "_name_key"], kind="mergesort", na_position="last")
        .reset_index(drop=True)
    )

    pr_fixed_order = pr_fixed_order or []
    pr_fixed_set = set(pr_fixed_order)

    result = []
    for uf, g in out.groupby(uf_col, sort=False, dropna=False):
        if not isinstance(uf, str) or uf.strip() == "":
            result.append(g)
            continue

        is_blocked = uf in blocked_ufs
        is_sp = uf == "SP"
        is_df = uf == "DF"
        is_pr = uf == "PR"

        gg = g.copy()
        existing_nums = _parse_existing_nums(gg[id_col], uf)
        max_existing = int(existing_nums.max()) if not existing_nums.dropna().empty else 0

        if is_pr and pr_fixed_set:
            is_fixed_station = gg[station_col].isin(pr_fixed_set)
        else:
            is_fixed_station = pd.Series(False, index=gg.index)

        if is_blocked:
            can_fill = pd.Series(False, index=gg.index)
        else:
            can_fill = gg[id_col].isna() | gg[id_col].str.strip().eq("")
            if is_pr and pr_fixed_set:
                can_fill = can_fill & ~is_fixed_station

        idx_to_fill = gg.index[can_fill]
        if len(idx_to_fill) > 0:
            start_n = max_existing + 1
            seq = pd.Series(range(start_n, start_n + len(idx_to_fill)), index=idx_to_fill)
            new_ids = uf + seq.astype(int).astype(str).str.zfill(4)

            while gg[id_col].isin(new_ids).any():
                start_n += 1
                seq = pd.Series(range(start_n, start_n + len(idx_to_fill)), index=idx_to_fill)
                new_ids = uf + seq.astype(int).astype(str).str.zfill(4)

            gg.loc[idx_to_fill, id_col] = new_ids

        result.append(gg)

    out2 = pd.concat(result, axis=0).sort_index()
    return out2.drop(columns=["_name_key", "_start_key"], errors="ignore")

In [785]:
pr_ordem_fixa = [
    "CIC","STC","ASS","BOQ","CSN","PAR","UEG","RPR","SIX","CAS",
    "PGA","LON","MRGA","FOZ","CVEL"
]

blocked = ("RJ","ES","SC","PE","PB","MA","CE","BA")  # manter como está

for uf, d in ufs_dfs.items():
    ufs_dfs[uf] = assign_id_mma_all_rules(
        d,
        uf_col="UF",
        start_col="INICIO",
        station_col="ID_OEMA",
        id_col="ID_MMA",
        anchor="start",
        pr_fixed_order=pr_ordem_fixa,
        blocked_ufs=blocked
    )

pr_fixed_map = {
    "CIC":"PR0001","STC":"PR0002","ASS":"PR0003","BOQ":"PR0004","CSN":"PR0005",
    "PAR":"PR0006","UEG":"PR0007","RPR":"PR0008","SIX":"PR0009","CAS":"PR0010",
    "PGA":"PR0011","LON":"PR0012","MRGA":"PR0013","FOZ":"PR0014","CVEL":"PR0015"
}

def aplicar_map_pr_inplace(df, uf_col="UF", nome_col="ID_OEMA", id_col="ID_MMA"):
    if id_col not in df.columns:
        df[id_col] = pd.NA
    mask = (df[uf_col].astype(str).str.upper().eq("PR")) & (df[id_col].isna() | df[id_col].astype(str).str.strip().eq(""))
    df.loc[mask, id_col] = df.loc[mask, nome_col].map(pr_fixed_map)

for name, d in ufs_dfs.items():
    aplicar_map_pr_inplace(d, uf_col="UF", nome_col="ID_OEMA", id_col="ID_MMA")
    print(f"\n=== {name} ===")
    display(d)


=== AC_estacoes ===


/tmp/ipykernel_272782/291595284.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s = pd.to_datetime(series, errors="coerce", utc=True)
/tmp/ipykernel_272782/291595284.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  base = pd.to_datetime(raw[mask_ym] + "-01", errors="coerce", utc=True)
/tmp/ipykernel_272782/291595284.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s = pd.to_datetime(series, errors="coerce", utc=True)
/tmp/ipykernel_272782/291595284.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`.

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,AC,AcreBioClima - UFAC,Rio Branco,AC0001,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,NaT,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
1,AC,Ministério Público do Estado do Acre (SEDE),Rio Branco,AC0002,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,NaT,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
2,AC,MPAC_ABR_01_promotoria,Assis Brasil,AC0003,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,NaT,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
3,AC,MPAC_ABR_02_SEMSA,Assis Brasil,AC0004,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,NaT,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
4,AC,MPAC_ACL_01_promotoria,Acrelandia,AC0005,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,NaT,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
5,AC,MPAC_BJR_01_promotoria,Bujari,AC0006,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,NaT,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
6,AC,MPAC_BRL_01_promotoria,Brasiléia,AC0007,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,NaT,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
7,AC,MPAC_BRL_02_radio fm 90.3,Brasiléia,AC0008,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,NaT,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
8,AC,MPAC_CPX_01_qpm,Capixaba,AC0009,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,NaT,2023.0,NaN,NaN,NaN,NaN,NaN,NaN
9,AC,MPAC_CZS_02_ciosp,Cruzeiro do Sul,AC0010,NaN,MP25,NaN,NaN,12.0,NaN,...,NaN,NaN,NaT,2023.0,NaN,NaN,NaN,NaN,NaN,NaN



=== AL_estacoes ===


,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,AL,[A303] MACEIÓ,Maceio,AL0001,NaN,"PTS,MP10,MP25,03,CO,SO2,NO2,FMC",NaN,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AL,Braskem,Maceio,AL0002,NaN,"PTS,MP10,MP25,SO2,NO2,NO,NOX,O3,CO",NaN,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AL,Fabrica frango favorito,Santa Luzia do Norte,AL0003,NaN,"PTS,MP10,MP25,SO2,NO2,CO,O3",NaN,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AL,Frigorifico caprisu,Santa Luzia do Norte,AL0004,NaN,"PTS,MP10,MP25,SO2,NO2,O3,CO",NaN,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AL,QAR 04 - PLANTA DE BENEFICIAMENTO,Craibas,AL0005,NaN,"PTS,MP10,MP25,SO2",NaN,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,AL,QAR01-PAU FERRO,Craibas,AL0006,NaN,"PTS,MP10,MP25,SO2",NaN,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,AL,QAR02-LAGOA DA CRUZ,Craibas,AL0007,NaN,"PTS,MP10,MP25,SO2",NaN,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,AL,QAR03-LAGOA DO MEL,Craibas,AL0008,NaN,"PTS,MP10,MP25,SO2",NaN,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,<NA>,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,<NA>,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


...

,UF,COD_UF_IBGE,ID_OEMA,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,PROP_ENTIDADE,...,FIM,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA
0,RS,43.0,Canoas VCOMAR,RS0004,Canoas,4304606.0,Referencia,Automatica,FEPAM,Pública,...,2009-12-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala de Bairro
1,RS,43.0,Sapucaia,RS0001,Sapucaia do Sul,4320008.0,Referencia,Automatica,FEPAM,Pública,...,2010-01-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala urbana
2,RS,43.0,Canoas P Universitário,RS0005,Canoas,4304606.0,Referencia,Automatica,Refap,Pública,...,-,Ativa,A cada 1 mês,NaN,Sim,Consulta 2025,NaN,Sim,Sim,Escala de Bairro
3,RS,43.0,Esteio Vila Ezequiel,RS0006,Canoas,4304606.0,Referencia,Automatica,Refap,Pública,...,2019-12-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala urbana
4,RS,43.0,Gravataí C Jardim Timbaúva,RS0012,Gravataí,4309209.0,Referencia,Automatica,FEPAM,Pública,...,-,Ativa,A cada 2 meses,Checagem mensal,Sim,Consulta 2025,NaN,Sim,Sim,Escala urbana
5,RS,43.0,Charqueadas-AT,RS0013,Charqueadas,4305355.0,Referencia,Automatica,Tractebel,Privada,...,2018-12-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala urbana
6,RS,43.0,Triunfo DEPREC,RS0009,Triunfo,4322004.0,Referencia,Automatica,Tractebel,Privada,...,2018-12-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala urbana
7,RS,43.0,Triunfo DEPREC,RS0009,Triunfo,4322004.0,Referencia,Automatica,Tractebel,Privada,...,2018-12-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala urbana
8,RS,43.0,Triunfo DEPREC,RS0009,Triunfo,4322004.0,Referencia,Automatica,Tractebel,Privada,...,2018-12-01 00:00:00,Inativa,A cada 1 mês,NaN,Não,Consulta 2025,NaN,Sim,Sim,Escala urbana
9,RS,43.0,Triunfo Polo Movel,RS0016,Triunfo,4322004.0,Referencia,Automatica,Braskem,Privada,...,-,Ativa,A cada 1 mês,NaN,Sim,Consulta 2025,NaN,Sim,Sim,Escala urbana



=== SC_estacoes ===


,UF,COD_UF_IBGE,ID_OEMA,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,PROP_ENTIDADE,...,FIM,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA
0,SC,42,Capivari,SC0002,Capivari de Baixo,4203956,Referencia,Automatica,Diamante Geração De EnergiaLTDA.,Privada,...,NaN,Ativa,NaN,NaN,Sim,Consulta 2025,NaN,NaN,NaN,NaN
1,SC,42,São Bernardo,SC0003,Tubarão,4218707,Referencia,Automatica,Diamante Geração De EnergiaLTDA.,Privada,...,NaN,Ativa,NaN,NaN,Sim,Consulta 2025,NaN,NaN,NaN,NaN
2,SC,42,UFSC,SC0004,Florianopolis,4205407,Referencia,Automatica,IMA,Publica,...,NaN,Ativa,"Mensal(O3),Mensal(NO2),Mensal(MP25)",NaN,Não,Consulta 2025,NaN,NaN,NaN,NaN
3,SC,42,Vila Moema,SC0001,Tubarão,4218707,Referencia,Automatica,Diamante Geração De EnergiaLTDA.,Privada,...,NaN,Ativa,NaN,NaN,Sim,Consulta 2025,NaN,NaN,NaN,NaN



=== SP_estacoes ===


,ID_OEMA,CALIBRACAO,CATEGORIA,CD_MUN,CIDADE,COD_UF_IBGE,FIM,FINALIDADE,FONTE,FUNCIONAMENTO,...,OP_ENTIDADE,POLUENTE,PROPRIETARIO,PROP_ENTIDADE,REALOCACAO,REP_ESPACIAL,REP_ESPACIAL_DECLARADA,STATUS,UF,RECONHECIDA
0,Cerqueira César,diária,Referencia,3550308.0,São Paulo,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,NO,CO,SO2,NO2,NOX,MP25",CETESB,Pública,Não,NaN,Micro,Ativa,SP,NaN
1,Congonhas,diária,Referencia,3550308.0,São Paulo,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,NO,CO,O3,SO2,NO2,NOX,MP25",CETESB,Pública,Não,NaN,Micro,Ativa,SP,NaN
2,Cubatão-Centro,diária,Referencia,3513504.0,Cubatão,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,NO,BENZENO,O3,SO2,NO2,TOLUENO,NOX,MP25",CETESB,Pública,Não,NaN,Bairro,Ativa,SP,NaN
3,Cubatão-V.Parisi,diária,Referencia,3513504.0,Cubatão,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,NO,O3,SO2,NO2,NOX",CETESB,Pública,Não,NaN,Bairro,Ativa,SP,NaN
4,Diadema,diária,Referencia,3513801.0,Diadema,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,O3",CETESB,Pública,Não,NaN,Bairro,Ativa,SP,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,S.André-Centro,NaN,Referencia,2513851.0,Santo André,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,CO",CETESB,Pública,NaN,NaN,NaN,NaN,SP,NaN
98,S.André-Centro,NaN,Referencia,3547809.0,Santo André,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,CO",CETESB,Pública,NaN,NaN,NaN,NaN,SP,NaN
99,S.André-Paço Municipal,NaN,Referencia,2513851.0,Santo André,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,CO",CETESB,Pública,NaN,NaN,NaN,NaN,SP,NaN
100,S.André-Paço Municipal,NaN,Referencia,3547809.0,Santo André,35.0,NaN,Fornecer dados,Coleta Interna,Automatica,...,Pública,"MP10,CO",CETESB,Pública,NaN,NaN,NaN,NaN,SP,NaN


2. Limpar e padronizar caracteres

In [119]:
# for name, d in ufs_dfs.copy().items():
#     dfs_clean = cuf.clean_df_all_text(d, in_place=True)

3. Unir DFs 

In [786]:
uf_frame, uf_conflicts = cuf.merge_by_id_multi(
    ufs_dfs.copy(),
    id_cols=['ID_OEMA','POLUENTE'],
    source_priority=None,
    drop_empty_strings=True,
)

In [787]:
uf_frame

,ID_OEMA,POLUENTE,UF,CIDADE,ID_MMA,ID_MMA_COMPLETO,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA,OPERACAO,REP_ESPACIAL
0,AcreBioClima - UFAC,MP25,AC,Rio Branco,AC0001,<NA>,<NA>,<NA>,12.0,<NA>,...,<NA>,2023.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,Ministério Público do Estado do Acre (SEDE),MP25,AC,Rio Branco,AC0002,<NA>,<NA>,<NA>,12.0,<NA>,...,<NA>,2023.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,MPAC_ABR_01_promotoria,MP25,AC,Assis Brasil,AC0003,<NA>,<NA>,<NA>,12.0,<NA>,...,<NA>,2023.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,MPAC_ABR_02_SEMSA,MP25,AC,Assis Brasil,AC0004,<NA>,<NA>,<NA>,12.0,<NA>,...,<NA>,2023.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,MPAC_ACL_01_promotoria,MP25,AC,Acrelandia,AC0005,<NA>,<NA>,<NA>,12.0,<NA>,...,<NA>,2023.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403,Mogi das Cruzes,"MP10,NO,O3,NO2,NOX",SP,Mogi das Cruzes,SP0287,<NA>,<NA>,3530607.0,35.0,CETESB,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
404,Paulínia Sul,"MP10,NO,O3,SO2,NO2,NOX",SP,Paulínia,SP0112,<NA>,<NA>,3536505.0,35.0,CETESB,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
405,Pirassununga-EM,"MP10,NO,O3,NO2,NOX",SP,Pirassununga,SP0268,<NA>,<NA>,3539301.0,35.0,CETESB,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
406,S.André-Centro,"MP10,CO",SP,Santo André,SP0101,<NA>,<NA>,2513851.0,35.0,CETESB,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


4. Conferir nomes de poluentes e substituir pelo dicionário quando diferente (ex. PM10 = MP10)

In [788]:
def _erase(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def _norm(s: str) -> str:
    s = str(s)
    s = re.sub(r"\(.*?\)|\[.*?\]", "", s).replace("µ", "u")
    s = _erase(s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def build_cod_map(df_cod, col_sinonimos=None):
    m = {}
    for _, r in df_cod.iterrows():
        m[_norm(r["POLUENTE"])] = r["NOME_PASTA"]
        if col_sinonimos and col_sinonimos in df_cod.columns and pd.notna(r[col_sinonimos]):
            for alt in str(r[col_sinonimos]).split("|"):
                alt = alt.strip()
                if alt:
                    m[_norm(alt)] = r["NOME_PASTA"]
    return m

def normalize_pols_cell(text, cod_map, sep=r"[;,/|]+"):
    if pd.isna(text):
        return ""
    parts = re.split(sep, str(text))
    seen, out = set(), []
    for p in parts:
        k = _norm(p)
        name = cod_map.get(k)
        if name and name not in seen:
            seen.add(name); out.append(name)
    return ",".join(sorted(out))

# uso linha a linha
cod_map = build_cod_map(df_cod)  # faça uma vez
uf_frame["POLUENTE"] = uf_frame["POLUENTE"].apply(lambda s: normalize_pols_cell(s, cod_map))

In [789]:
def limpar_ipynb_poluente(df, col="POLUENTE"):
    s = df[col].astype(str)

    # remove tokens contendo ".ipynb_checkpoints"
    s = s.str.replace(r'(?i)(^|,)\s*[^,]*ipynb[_-]?checkpoints[^,]*(?=,|$)', '', regex=True)
    # remove tokens contendo ".ipynb"
    s = s.str.replace(r'(?i)(^|,)\s*[^,]*\.ipynb[^,]*(?=,|$)', '', regex=True)

    # compacta vírgulas e espaços
    s = s.str.replace(r'\s*,\s*', ',', regex=True)
    s = s.str.replace(r',+', ',', regex=True).str.strip(', ')

    # dedup dos itens mantendo a ordem
    def _dedup(v):
        if not v:
            return v
        parts = [p for p in v.split(',') if p]
        seen = set()
        out = []
        for p in parts:
            if p not in seen:
                seen.add(p)
                out.append(p)
        return ",".join(out)

    df[col] = s.map(_dedup)
    return df

In [790]:
def _to_ascii_upper(s: str) -> str:
    # remove acentos e normaliza subscrito/sobrescrito para dígitos
    subs = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    sups = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.translate(subs).translate(sups).upper().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def _map_token(t: str) -> list[str]:
    """Mapeia um token para 0..n nomes padronizados."""
    t0 = _to_ascii_upper(t)

    # descartes
    if t0 in {"", "NAN", "NA", "NONE", "NULL", "ODOR", "ODORES"}:
        return []

    # PM/MP normalização
    t1 = t0.replace("PM", "MP")  # PM10 -> MP10, PM2.5 -> MP2.5
    t1 = t1.replace("MP1O", "MP10")  # O no lugar de zero

    # MP2.5 variações -> MP25
    if re.fullmatch(r"MP2([.,/]?5)?|MP25|PM2([.,/]?5)?", t0):
        return ["MP25"]

    if t1 in {"MP10"}:
        return ["MP10"]
    if t1 in {"MP1"}:
        return ["MP1"]

    # abreviações comuns
    direct = {
        "NOX": "NOX",
        "NO": "NO",
        "NO2": "NO2",
        "O3": "O3",
        "SO2": "SO2",
        "CO": "CO",
        "CH4": "CH4",
        "PTS": "PTS",
        "VOC": "VOC",
        "HCT": "HCT",
        "HCNM": "HCNM",
        "ERT": "ERT",
        "FMC": "FMC",
        "H2S": "H2S",
        "BENZENO": "BENZENO",
        "TOLUENO": "TOLUENO",
        "ETILBENZENO": "ETILBENZENO",
        "XILENO": "XILENO",
        "OXILENO": "OXILENO",
        "MPXILENO": "MPXILENO",
        "ACETAL": "ACETAL",
        "FORMAL": "FORMAL",
        "CH2O": "FORMAL",   # formaldeído
        "BEN": "BENZENO",
        "BENZ": "BENZENO",
        "TOL": "TOLUENO",
        "ETBEN": "ETILBENZENO",
        "ETILBEN": "ETILBENZENO",
        "MPXIL": "MPXILENO",
        "OXIL": "OXILENO",
        "XIL": "XILENO",
    }
    if t1 in direct:
        return [direct[t1]]

    # caso especial “ETIL,ORTO” já será quebrado pelo split; mas se vier colado:
    if t1 in {"ETIL,ORTO", "ETIL ORTO"}:
        return ["ETILBENZENO", "OXILENO"]

    # fallback: mantemos o token bruto normalizado
    return [t1]

def normalizar_coluna_poluente(df: pd.DataFrame, df_cod: pd.DataFrame, col="POLUENTE") -> pd.DataFrame:
    if col not in df.columns:
        return df.copy()

    # vocabulário permitido
    allowed = set(df_cod["NOME_PASTA"].astype(str).str.upper().str.strip())

    # separadores: vírgula, ;, /, |, e “ e ”
    sep_re = re.compile(r"\s*(?:,|;|/|\||\se\s)\s*", flags=re.I)

    def process_cell(val) -> str:
        if pd.isna(val):
            return ""
        tokens = []
        seen = set()
        # quebra
        parts = [p for p in sep_re.split(str(val)) if p.strip()]
        for p in parts:
            mapped = _map_token(p)
            for m in mapped:
                if m in allowed and m not in seen:
                    seen.add(m)
                    tokens.append(m)
        return ",".join(tokens)

    out = df.copy()
    out[col] = out[col].apply(process_cell)
    return out

def listar_tokens_restantes(df: pd.DataFrame, col="POLUENTE") -> list[str]:
    parts = df[col].astype(str).str.split(",").explode().dropna().str.strip()
    return sorted([t for t in parts.unique() if t])

# -------------------- uso --------------------
# df_cod precisa ter a coluna NOME_PASTA com os nomes oficiais
# df_merged é o seu DF com a coluna POLUENTE

uf_frame = limpar_ipynb_poluente(uf_frame, col="POLUENTE")
uf_frame = normalizar_coluna_poluente(uf_frame, df_cod, col="POLUENTE")
print(listar_tokens_restantes(uf_frame, "POLUENTE"))

['ACETAL', 'BENZENO', 'CH4', 'CO', 'ERT', 'ETILBENZENO', 'FMC', 'FORMAL', 'H2S', 'HCNM', 'HCT', 'MP1', 'MP10', 'MP25', 'NO', 'NO2', 'NOX', 'O3', 'PTS', 'SO2', 'TOLUENO', 'XILENO']


5. Conferir linhas repetidas ou informações diferentes

In [791]:
# 1) Linhas em que ID_OEMA se repetem
dups = uf_frame[uf_frame["ID_OEMA"].duplicated(keep=False)].sort_values("ID_OEMA")
print(dups)

# 2) Todos os IDs duplicados
dup_ids = uf_frame["ID_OEMA"][uf_frame["ID_OEMA"].duplicated()].unique()
print("Duplicated IDs:", dup_ids)

# 3) Quantas IDs foram repetidas
counts = uf_frame["ID_OEMA"].value_counts()
print(counts[counts > 1])

Empty DataFrame
Columns: [ID_OEMA, POLUENTE, UF, CIDADE, ID_MMA, ID_MMA_COMPLETO, COD_POLUENTE, CD_MUN, COD_UF_IBGE, PROPRIETARIO, PROP_ENTIDADE, OPERADOR, OP_ENTIDADE, LATITUDE, LONGITUDE, MOBILIDADE, CATEGORIA, FUNCIONAMENTO, METODO, MARCA, FINALIDADE, MONITORAR, FONTE, CALIBRACAO, REALOCACAO, OBS_CALIBRACAO, INICIO, FIM, DADOS_MONITORAMENTO, RECONHECIDA, OBS_GERAIS, CERTIFICACAO, STATUS, REP_ESPACIAL_DECLARADA, OPERACAO, REP_ESPACIAL]
Index: []

[0 rows x 36 columns]
Duplicated IDs: []
Series([], Name: count, dtype: int64)


##### ID_MMA_COMPLETO intermediário

In [796]:
df_full = explode_and_apply_PolCod(uf_frame, df_cod)

In [797]:
from typing import Optional

def _str_no_dotzero(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

def _cod_poluente_clean(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    s_num = pd.to_numeric(s, errors="coerce")
    out = _str_no_dotzero(s)
    m = s_num.notna()
    out.loc[m] = s_num.loc[m].astype("Int64").astype(str).str.zfill(3)
    return out.fillna("")

def _extract_uf_num(idmma: pd.Series) -> pd.DataFrame:
    s = idmma.astype("string").str.strip()
    uf = s.str.extract(r"^([A-Z]{2})", expand=False)
    num = s.str.extract(r"^[A-Z]{2}\s*0*?(\d+)$", expand=False)
    num = pd.to_numeric(num, errors="coerce").astype("Int64")
    return pd.DataFrame({"_UF": uf, "_NUM": num}, index=s.index)

# >>> ANOTAÇÃO CORRIGIDA AQUI <<<
def _pair_for_row(uf: Optional[str], num: Optional[int]) -> str:
    if uf is None or pd.isna(uf):
        return ""
    uf = str(uf).upper()

    fixed_default = {
        "SP": "RA", "ES": "RA", "MG": "RA", "SC": "RA", "RS": "RA",
        "PR": "RA", "BA": "ND", "MA": "RA", "MT": "IA", "PE": "ND",
        "RR": "RS", "PB": "ND", "CE": "ND",
    }

    def ge(n: Optional[int], cutoff: int) -> bool:
        return n is not None and not pd.isna(n) and int(n) >= cutoff

    if uf == "RJ":
        return "ND" if ge(num, 1000) else "RA"
    if uf == "PA":
        return "ND" if ge(num, 1001) else "RA"
    if uf == "RN":
        return "ND" if ge(num, 1001) else "RA"

    return fixed_default.get(uf, "RA")

def make_id_mma_completo_fixos(df: pd.DataFrame) -> pd.DataFrame:
    out   = df.copy()
    idmma = _str_no_dotzero(out.get("ID_MMA", pd.Series(index=out.index)))
    cod   = _cod_poluente_clean(out.get("COD_POLUENTE", ""))

    meta = _extract_uf_num(idmma)
    uf, num = meta["_UF"], meta["_NUM"]

    pair = pd.Series((_pair_for_row(u, int(n) if not pd.isna(n) else None) for u, n in zip(uf, num)),
                     index=out.index, dtype="string")

    out["ID_MMA_COMPLETO"] = (idmma.fillna("") + pair.fillna("") + cod).astype("string")
    return out

In [800]:
df_full = make_id_mma_completo_fixos(df_full)

6. Comparar planilha final com planilha PurpleAir

In [801]:
# Importar planilha com dados do PurpleAir coletados internamente 
base = Path.cwd().parent  
out_dir = base / "data" / "DADOS_ESTACOES" / "Indicativas" 
out_dir.mkdir(parents=True, exist_ok=True)

df_purple = pd.read_csv(out_dir / 'Compiled_PurpleAirStations.csv')

In [805]:
df_purple

,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,FIM,LATITUDE,LONGITUDE,MONITORAR,FONTE,CERTIFICACAO,COD_UF_IBGE,ANOS_MONITORADOS,BASE_DADOS,ELEVACAO
0,MT,NaN,NaN,PELD-TANG,NaN,NaN,NaN,Publico,NaN,NaN,...,2025-09-19 10:43:38-03:00,-13.062653,-52.380970,NaN,PurpleAir 2025,NaN,51.0,2025,NaN,1190.0
1,SP,São Paulo,NaN,Vielas da Água Preta,NaN,NaN,NaN,Publico,NaN,NaN,...,2025-09-19 10:44:50-03:00,-23.536737,-46.692394,NaN,PurpleAir 2025,NaN,35.0,2025,NaN,2435.0
2,MT,NaN,NaN,DIAMANTINO QUALIDADE DO AR,NaN,NaN,NaN,Publico,NaN,NaN,...,2025-09-19 10:43:18-03:00,-14.399174,-56.436085,NaN,PurpleAir 2025,NaN,51.0,2025,NaN,950.0
3,AC,NaN,NaN,MPAC_PTA_01_Sec.infraestrutura,NaN,NaN,NaN,Publico,NaN,NaN,...,2025-09-19 10:43:48-03:00,-9.727080,-67.698020,NaN,PurpleAir 2025,NaN,12.0,"2019,2020,2021,2022,2023,2024,2025",NaN,677.0
4,AC,NaN,NaN,MPAC_FIJ_01_promotoria,NaN,NaN,NaN,Publico,NaN,NaN,...,2025-09-19 10:43:44-03:00,-8.170079,-70.355030,NaN,PurpleAir 2025,NaN,12.0,"2019,2020,2021,2022,2023,2024,2025",NaN,525.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,DF,NaN,NaN,UnB Odisseia Cafuringa,NaN,NaN,NaN,Publico,NaN,NaN,...,2025-09-19 10:44:12-03:00,-15.541478,-47.971092,NaN,PurpleAir 2025,NaN,53.0,"2024,2025",NaN,2801.0
173,DF,Gama,NaN,UnB Odisseia Gama,NaN,NaN,NaN,Publico,NaN,NaN,...,2025-09-19 10:44:35-03:00,-15.989608,-48.045723,NaN,PurpleAir 2025,NaN,53.0,"2024,2025",NaN,3989.0
174,RJ,Rio de Janeiro,NaN,RIO401101,NaN,NaN,NaN,Publico,NaN,NaN,...,2025-09-18 15:20:04-03:00,-22.911589,-43.756820,NaN,PurpleAir 2025,NaN,33.0,"2024,2025",NaN,2.0
175,MT,Cuiabá,NaN,UNIC,NaN,NaN,NaN,Publico,NaN,NaN,...,2025-09-19 10:43:51-03:00,-15.623796,-56.086617,NaN,PurpleAir 2025,NaN,51.0,2025,NaN,514.0


In [806]:
df_purple_full = explode_and_apply_PolCod(df_purple, df_cod)
df_purple_full = df_purple_full[df_purple_full["POLUENTE"] == "MP25"].copy()

In [807]:
df_mma = df_full.copy()

In [808]:
# LATITUDE
df_mma['LATITUDE'] = (
    pd.to_numeric(df_mma['LATITUDE'].astype(str).str.replace(',', '.'), errors='coerce'))
# LONGITUDE
df_mma['LONGITUDE'] = (
    pd.to_numeric(df_mma['LONGITUDE'].astype(str).str.replace(',', '.'), errors='coerce'))

In [809]:
def _collapse_dupe_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Coalesce valores entre colunas duplicadas com o mesmo nome (esq→dir) e retorna colunas únicas."""
    df = df.copy()
    uniques = pd.Index(df.columns).unique()
    out = pd.DataFrame(index=df.index)
    for c in uniques:
        block = df.loc[:, df.columns == c]
        out[c] = block.bfill(axis=1).iloc[:, 0]  # primeiro não-nulo
    return out

def _norm_basic(s: pd.Series) -> pd.Series:
    """Upper, trim, string; vazio vira ''."""
    return s.astype("string").fillna("").str.strip().str.upper()

def _norm_name(s: pd.Series) -> pd.Series:
    """
    Normalização forte para nomes:
    - strip, upper
    - remove acentos
    - troca pontuação por espaço
    - colapsa múltiplos espaços
    """
    t = s.astype("string").fillna("").str.strip().str.upper()
    # remove acentos
    t = t.map(lambda x: "".join(ch for ch in ud.normalize("NFKD", x) if not ud.combining(ch)))
    # troca pontuação por espaço
    t = t.str.replace(r"[^A-Z0-9]+", " ", regex=True)
    # colapsa espaços
    t = t.str.replace(r"\s+", " ", regex=True).str.strip()
    return t

def _nan_if_empty(s: pd.Series) -> pd.Series:
    """Converte '' (ou só espaços) em NA para coalescência correta."""
    s = s.astype("string")
    return s.mask(s.str.strip().eq(""))

# ---------------------------
# Merge principal
# ---------------------------

def merge_by_df(
    df_purple: pd.DataFrame,
    df_mma: pd.DataFrame,
    key_cols=("UF", "ID_OEMA", "POLUENTE"),
    mma_priority=True  # se True, df_mma prevalece quando ambos têm valor
) -> pd.DataFrame:
    # 1) tratar duplicadas
    p = _collapse_dupe_cols(df_purple)
    m = _collapse_dupe_cols(df_mma)

    # 2) chaves normalizadas para juntar
    #    - ID_OEMA: normalização forte (_norm_name)
    #    - UF, POLUENTE: normalização básica (_norm_basic)
    def _norm_for_key(colname: str, s: pd.Series) -> pd.Series:
        if colname.upper() == "ID_OEMA":
            return _norm_name(s)
        else:
            return _norm_basic(s)

    p_keys = [f"{k}__norm" for k in key_cols]
    m_keys = [f"{k}__norm" for k in key_cols]
    for k, pk, mk in zip(key_cols, p_keys, m_keys):
        p[pk] = _norm_for_key(k, p.get(k, pd.Series(index=p.index)))
        m[mk] = _norm_for_key(k, m.get(k, pd.Series(index=m.index)))

    # 3) descobrir colunas compartilhadas e exclusivas
    shared = sorted(set(p.columns).intersection(m.columns) - set(key_cols) - set(p_keys) - set(m_keys))
    only_p = [c for c in p.columns if c not in m.columns and c not in p_keys]
    only_m = [c for c in m.columns if c not in p.columns and c not in m_keys]

    # 4) outer merge nas chaves normalizadas
    merged = p.merge(
        m, left_on=p_keys, right_on=m_keys, how="outer",
        suffixes=("_purple", "_mma"), indicator=True
    )

    # 5) construir colunas finais
    out = pd.DataFrame(index=merged.index)

    # chaves finais: se houver conflito, priorizar MMA (mais confiável)
    for k in key_cols:
        mma_k = _nan_if_empty(merged.get(f"{k}_mma"))
        pur_k = _nan_if_empty(merged.get(f"{k}_purple"))
        out[k] = mma_k.combine_first(pur_k)

    # colunas compartilhadas
    for c in shared:
        mma_c = _nan_if_empty(merged[f"{c}_mma"])
        pur_c = _nan_if_empty(merged[f"{c}_purple"])
        out[c] = (mma_c.combine_first(pur_c)) if mma_priority else (pur_c.combine_first(mma_c))

    # colunas só do purple
    for c in only_p:
        if f"{c}_purple" in merged:
            out[c] = merged[f"{c}_purple"]
        elif c in merged:
            out[c] = merged[c]

    # colunas só do mma
    for c in only_m:
        if f"{c}_mma" in merged:
            out[c] = merged[f"{c}_mma"]
        elif c in merged:
            out[c] = merged[c]

    # 6) status da correspondência
    out["RECONHECIDA"] = merged["_merge"].map({
        "both": "Reconhecida",
        "left_only": "Nao reconhecida",
        "right_only": ""
    })

    # 7) ordem de colunas: todas do purple + extras do mma + RECONHECIDA
    out_cols = list(p.columns.drop(p_keys)) + [c for c in out.columns if c not in p.columns]
    out = out[[c for c in out_cols if c in out.columns]]

    return out

In [810]:
print(len(df_purple_full))
print(len(df_mma))

176
2203


In [814]:
dfs_merged = merge_by_df(df_purple_full.copy(), df_mma.copy())
dfs_merged

,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,ANOS_MONITORADOS,BASE_DADOS,ELEVACAO,REALOCACAO,OBS_CALIBRACAO,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,REP_ESPACIAL_DECLARADA,OPERACAO
0,<NA>,Cobija,<NA>,Plaza Principal Cobija,<NA>,<NA>,<NA>,Publico,<NA>,<NA>,...,"2022,2023,2024,2025",NaN,675.0,NaN,NaN,NaN,Nao reconhecida,NaN,NaN,NaN
1,AC,Rio Branco,<NA>,AcreBioClima - UFAC,AC0001,AC0001RA002,<NA>,Publico,<NA>,<NA>,...,"2019,2020,2021,2022,2023,2024,2025",NaN,524.0,<NA>,<NA>,<NA>,Reconhecida,<NA>,<NA>,<NA>
2,AC,Rio Branco,<NA>,AcreBioClima - UFAC,AC0001,AC0001RA002,<NA>,Publico,<NA>,<NA>,...,"2022,2023,2024,2025",NaN,524.0,<NA>,<NA>,<NA>,Reconhecida,<NA>,<NA>,<NA>
3,AC,Rio Branco,<NA>,Ministério Público do Estado do Acre (SEDE),AC0002,AC0002RA002,<NA>,Publico,<NA>,<NA>,...,"2019,2020,2021,2022,2023,2024,2025",NaN,495.0,<NA>,<NA>,<NA>,Reconhecida,<NA>,<NA>,<NA>
4,AC,Assis Brasil,<NA>,MPAC_ABR_01_promotoria,AC0003,AC0003RA002,<NA>,Publico,<NA>,<NA>,...,"2019,2020,2021,2022,2023,2024,2025",NaN,783.0,<NA>,<NA>,<NA>,Reconhecida,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2362,SP,Taubaté,3554102.0,Taubaté,SP0280,SP0280RA003,CETESB,Pública,CETESB,Pública,...,NaN,NaN,NaN,Não,<NA>,<NA>,,<NA>,Bairro,<NA>
2363,SP,São Paulo,<NA>,Vielas da Água Preta,<NA>,<NA>,<NA>,Publico,<NA>,<NA>,...,2025,NaN,2435.0,NaN,NaN,NaN,Nao reconhecida,NaN,NaN,NaN
2364,TO,<NA>,<NA>,(IPAM) TI Xerente,<NA>,<NA>,<NA>,Publico,<NA>,<NA>,...,"2024,2025",NaN,1068.0,NaN,NaN,NaN,Nao reconhecida,NaN,NaN,NaN
2365,TO,Palmas,<NA>,MPTO_PMW_01_procuradoriageral,<NA>,<NA>,<NA>,Publico,<NA>,<NA>,...,"2020,2021,2022,2023,2024,2025",NaN,850.0,NaN,NaN,NaN,Nao reconhecida,NaN,NaN,NaN


In [731]:
# out_file = out_dir / "TESTE.csv"
# dfs_merged.to_csv(out_file, index=False, encoding="utf-8")
# print("Saved to:", out_file.resolve())

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/Indicativas/TESTE.csv


7. Explodir poluentes e preencher coluna com códigos - um poluente por linha

In [486]:
# def explode_pol_mtd_marca(df):
#     linhas = []

#     for _, row in df.iterrows():
#         # --- POLUENTES (with MP expansion) ---
#         poluentes = [p.strip() for p in str(row.get("POLUENTE", "")).split(",") if p.strip()]
#         poluentes_expandidos = []
#         for p in poluentes:
#             if p.upper() == "MP":
#                 poluentes_expandidos.extend(["MP10", "MP25", "PTS"])
#             else:
#                 poluentes_expandidos.append(p)

#         # Ensure string inputs
#         metodo_raw = "" if pd.isna(row.get("METODO", "")) else str(row.get("METODO", "")).strip()
#         marca_raw  = "" if pd.isna(row.get("MARCA", ""))  else str(row.get("MARCA", "")).strip()

#         # --- Parse METODO with optional per-pollutant mapping ---
#         metodos_dict = {}
#         if metodo_raw:
#             for item in metodo_raw.split(","):
#                 item = item.strip()
#                 match = re.search(r"\((.*?)\)", item)
#                 if match:
#                     pols = [pp.strip() for pp in match.group(1).split(",")]
#                     for p in pols:
#                         if p.upper() == "MP":
#                             for mp in ["MP10", "MP25", "PTS"]:
#                                 metodos_dict[mp] = item
#                         else:
#                             metodos_dict[p] = item

#         # If there is a single generic method (no commas and no parentheses), apply to all
#         metodo_default = ""
#         if metodo_raw and ("," not in metodo_raw) and (not re.search(r"\(.*\)", metodo_raw)):
#             metodo_default = metodo_raw

#         # --- Parse MARCA with optional per-pollutant mapping ---
#         marcas_dict = {}
#         if marca_raw:
#             for item in marca_raw.split(","):
#                 item = item.strip()
#                 match = re.search(r"\((.*?)\)", item)
#                 if match:
#                     pols = [pp.strip() for pp in match.group(1).split(",")]
#                     for p in pols:
#                         if p.upper() == "MP":
#                             for mp in ["MP10", "MP25", "PTS"]:
#                                 marcas_dict[mp] = item
#                         else:
#                             marcas_dict[p] = item

#         # If there is a single generic brand (no commas and no parentheses), apply to all
#         marca_default = ""
#         if marca_raw and ("," not in marca_raw) and (not re.search(r"\(.*\)", marca_raw)):
#             marca_default = marca_raw

#         # --- Build exploded rows ---
#         for pol in poluentes_expandidos:
#             nova = row.copy()
#             nova["POLUENTE"] = pol
#             nova["METODO"] = metodos_dict.get(pol, metodo_default)
#             nova["MARCA"] = marcas_dict.get(pol, marca_default)
#             linhas.append(nova)

#     return pd.DataFrame(linhas).reset_index(drop=True)

In [487]:
# def explode_and_apply_PolCod(df, df_cod):
#     for col in ["POLUENTE", "COD_POLUENTE", "NOME_PASTA"]:
#         if col in df_cod.columns:
#             df_cod[col] = df_cod[col].astype("string").str.strip()
            
#     # --- Carregar dicionário de poluentes ---
#     mapa_codigos = dict(zip(df_cod["POLUENTE"].str.strip(), df_cod["COD_POLUENTE"].str.strip()))
#     mapa_nome = dict(zip(df_cod["POLUENTE"].str.strip(), df_cod["NOME_PASTA"].str.strip()))

#     # Explodir POLUENTE
#     if "POLUENTE" in df.columns:
#         df = explode_pol_mtd_marca(df)

#     # Preencher COD_POLUENTE
#     df["COD_POLUENTE"] = df.get("POLUENTE", "").map(mapa_codigos).fillna("").astype(str)
#     df["COD_POLUENTE"] = df["COD_POLUENTE"].apply(lambda x: x.zfill(3) if x.isdigit() else x)

#     # Substituir POLUENTE pelo NOME_PASTA do dicionário
#     df["POLUENTE"] = df.get("POLUENTE", "").map(mapa_nome).fillna(df.get("POLUENTE", ""))

#     return df

In [488]:
# df_full = explode_and_apply_PolCod(dfs_merged, df_cod)

Nota: Como a base de dados do ano anterior possui os poluentes já explodidos por linha, fazer a comparação entre estações após explodir poluentes nas estações

In [815]:
path = Path.cwd().parent / "data" / "DADOS_ESTACOES" / "Dados_2024" / "Monitoramento_QAr_BR.csv"

def load_csv_robust(p: Path) -> pd.DataFrame:
    # 1) Auto-detect delimiter, UTF-8
    try:
        return pd.read_csv(p, sep=None, engine="python", encoding="utf-8")
    except Exception:
        pass
    # 2) Auto-detect delimiter, Windows-1252
    try:
        return pd.read_csv(p, sep=None, engine="python", encoding="cp1252")
    except Exception:
        pass
    # 3) Semicolon + cp1252 (very common in BR CSVs)
    try:
        return pd.read_csv(p, sep=";", engine="python", encoding="cp1252")
    except Exception:
        pass
    # 4) Comma + cp1252, skipping bad lines
    return pd.read_csv(p, sep=",", engine="python", encoding="cp1252", on_bad_lines="skip")

# Optional: detect if the file is actually an Excel saved with .csv extension
def load_tabular(p: Path) -> pd.DataFrame:
    with open(p, "rb") as f:
        sig = f.read(4)
    if sig == b"PK\x03\x04":  # XLSX signature
        return pd.read_excel(p, engine="openpyxl")
    return load_csv_robust(p)

df_old = load_tabular(path)

In [816]:
df_full = dfs_merged.copy() 

In [819]:
print(len(df_old))
print(len(df_full))

2243
2367


In [820]:
id_col = "ID_OEMA"

def norm(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip().str.upper()

def overlap(df_full, df_old):
    # normalized ID columns
    mma_id = norm(df_full[id_col])
    pur_id = norm(df_old[id_col])
    
    # IDs present in both dataframes
    common_ids = pd.Index(mma_id.dropna()).intersection(pur_id.dropna())
    print("IDs present in both:", len(common_ids))
    
    # rows involved on each side
    mma_overlap = df_full[mma_id.isin(common_ids)]
    pur_overlap = df_old[pur_id.isin(common_ids)]
    #print("Rows in df_full with overlapping ID_OEMA:", len(mma_overlap))
    #print("Rows in df_purple_exp with overlapping ID_OEMA:", len(pur_overlap))
    
    # summary count per ID showing how many times it appears in each df
    both = pd.concat(
        [
            pd.DataFrame({id_col: mma_id, "_src": "mma"}),
            pd.DataFrame({id_col: pur_id, "_src": "purple"}),
        ],
        ignore_index=True,
    )
    summary = (
        both.dropna(subset=[id_col])
            .groupby([id_col, "_src"]).size()
            .unstack(fill_value=0)
            .query("mma > 0 and purple > 0")
            .sort_index()
    )
    print("Overlapping unique IDs:", summary.shape[0])

overlap(df_full, df_old)

IDs present in both: 174
Overlapping unique IDs: 174


In [821]:
def fill_location_from_old(df_full, df_old):
    # ensure strings for matching
    df_full = df_full.copy()
    df_old = df_old.copy()
    df_full["ID_OEMA"] = df_full["ID_OEMA"].astype(str).str.strip()
    df_old["ID_OEMA"]  = df_old["ID_OEMA"].astype(str).str.strip()

    # subset of df_old with needed columns
    cols = ["ID_OEMA", "LONGITUDE", "LATITUDE", "CIDADE"]
    old_sub = df_old[cols].drop_duplicates(subset=["ID_OEMA"], keep="first")

    # merge to bring old info into full
    merged = df_full.merge(old_sub, on="ID_OEMA", how="left", suffixes=("", "_old"))

    # fill only where df_full is empty but df_old has data
    for col in ["LONGITUDE", "LATITUDE", "CIDADE"]:
        mask_full_empty = merged[col].isna() | merged[col].astype(str).str.strip().eq("")
        mask_old_nonempty = ~(merged[f"{col}_old"].isna() | merged[f"{col}_old"].astype(str).str.strip().eq(""))
        merged.loc[mask_full_empty & mask_old_nonempty, col] = merged.loc[
            mask_full_empty & mask_old_nonempty, f"{col}_old"
        ]

    # drop helper columns
    merged = merged.drop(columns=[c for c in merged.columns if c.endswith("_old")])
    return merged

In [824]:
df_full = fill_location_from_old(df_full.copy(), df_old)

In [825]:
df_full['LATITUDE'] = (
    pd.to_numeric(df_full['LATITUDE'].astype(str).str.replace(',', '.'), errors='coerce'))
# LONGITUDE
df_full['LONGITUDE'] = (
    pd.to_numeric(df_full['LONGITUDE'].astype(str).str.replace(',', '.'), errors='coerce'))

In [826]:
print(len(df_full))
df_full['ID_OEMA'].nunique()

2367


564

In [828]:
# Conferir se ainda existem linhas duplicadas para o mesmo conjunto de flags

cols = ["ID_OEMA", "ID_MMA", "ID_MMA_COMPLETO", "UF"]

# 1) Flag duplicates by the key combo
dup_mask = df_full.duplicated(subset=cols, keep=False)  # marks all rows in each dup set

# 2) See only the duplicate rows
dups = df_full.loc[dup_mask].sort_values(cols)

# 3) Counts per duplicate key
dup_counts = (
    dups.groupby(cols, dropna=False)
        .size()
        .reset_index(name="COUNT")
        .sort_values("COUNT", ascending=False)
)
# dups
# dup_counts

In [829]:
df_full = df_full.drop_duplicates(subset=cols, keep="first").copy()

In [682]:
# base = Path.cwd().parent  # .../RQAR_2025_book
# out_dir = base / "data" 
# out_dir.mkdir(parents=True, exist_ok=True)

# out_file = out_dir / "TESTE.csv"
# df_full.to_csv(out_file, index=False, encoding="utf-8")
# print("Saved to:", out_file.resolve())

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/TESTE.csv


8. Completar dados faltantes e padronizar respostas

In [749]:
for col in df_full.columns:
    s = df_full[col]
    vc = s.dropna().value_counts()
    print(vc) 

UF
RJ    742
SP    434
MG    355
ES    124
PR     99
BA     85
RS     84
AM     64
CE     52
MT     49
AL     46
MA     42
AC     30
MS     24
PA     23
SC     22
DF     22
PE     21
RR     19
PB     12
AP      6
RO      6
TO      3
Name: count, dtype: Int64
CIDADE
Rio de Janeiro     235
São Paulo          129
Itaborai            62
Macae               62
Duque de Caxias     62
                  ... 
Cordeirópolis        1
Guarujá              1
Jaboticabal          1
Itu                  1
Palmas               1
Name: count, Length: 209, dtype: Int64
CD_MUN
3304557.0    234
3550308.0    121
3301702.0     62
3301900.0     62
3302403.0     62
            ... 
3512407.0      1
3516200.0      1
3518701.0      1
3523909.0      1
3524303.0      1
Name: count, Length: 95, dtype: Int64
ID_OEMA
VR - Santa Cecilia                 16
VR - Retiro                        16
VR - Belmonte                      16
RJ - Adalgisa Nery                 16
RJ - Ilha de Paqueta               15
            

In [830]:
def replace_vals(df):
    # colunas alvo, só aplica se existirem
    cols = [c for c in ["CATEGORIA", "FUNCIONAMENTO", "STATUS", "RECONHECIDA", "MONITORAR", "FONTE", "CALIBRACAO" 
                       "PROP_ENTIDADE", "OP_ENTIDADE", "METODO", "MOBILIDADE", "FINALIDADE", "MONITORAR", 
                        "REALOCACAO", "REP_ESPACIAL_DECLARADA", "MARCA"] if c in df.columns]

    # regex para strings "vazias" comuns
    null_like = r'^\s*(na|n/a|none|null|nan|nat)?\s*$'

    # 1) normaliza vazios em todas as colunas alvo
    # for c in cols:
    #     df[c] = df[c].replace(null_like, pd.NA, regex=True).fillna("Nao declarado")

    # 2) mapeamentos específicos
    if "CATEGORIA" in df.columns:
        df["CATEGORIA"] = df["CATEGORIA"].replace({
            "N": "Nao declarado",
            "D": "Nao declarado",
            "Naodeclarado": "Nao declarado",
            "Meteorologica": "Nao declarado",
            "Meteorológica": "Nao declarado",
            "Nao Aplicavel": "Nao declarado",
            "CertificadaEPA": "Referencia",
            "Certificada EPA": "Referencia",
            "Equivalente": "Referencia",
            "Referência": "Referencia",
            "Não Declarada": "Nao declarado",
            "Não Aplicável": "Nao declarado"
        })

    if "PROP_ENTIDADE" in df.columns:
        df["PROP_ENTIDADE"] = df["PROP_ENTIDADE"].replace({
            "Público": "Publica",
            "Pública": "Publica",
            "Publico": "Publica"
        })

    if "OP_ENTIDADE" in df.columns:
        df["PROP_ENTIDADE"] = df["PROP_ENTIDADE"].replace({
            "-": "Nao declarado",
            "Pública": "Publica",
        })

    if "METODO" in df.columns:
        df["METODO"] = df["METODO"].replace({
            "-": "Nao declarado",
            "x": "Nao declarado",
        })

    if "FUNCIONAMENTO" in df.columns:
        df["FUNCIONAMENTO"] = df["FUNCIONAMENTO"].replace({
            "N": "Nao declarado",
            "D": "Nao declarado",
            "Autmatica": "Automatica",
            "Automatico": "Automatica",
            "Automático": "Automatica",
            "Não Declarada": "Nao declarado",
            "Automática": "Automatica",
            "Autmática": "Automatica"
        })

    if "STATUS" in df.columns:
        df["STATUS"] = df["STATUS"].replace({
            "Sim": "Ativa",
            "sim": "Ativa",
            "Não": "Inativa",
            "Nao": "Inativa",
            "nao": "Inativa",
            "NAO": "Inativa",
        })
        
    if "RECONHECIDA" in df.columns:
        df["RECONHECIDA"] = df["RECONHECIDA"].replace({
            "Nao declarado" : "Nao reconhecida", 
            "Não": "Nao reconhecida",
            "Nao": "Nao reconhecida",
            "nao": "Nao reconhecida",
            "NAO": "Nao reconhecida",
            "Nao reconhecido": "Nao reconhecida",
            "Naoreconhecida": "Nao reconhecida",
            "Sim": "Reconhecida"
        })

    if "REP_ESPACIAL_DECLARADA" in df.columns:
        df["REP_ESPACIAL_DECLARADA"] = df["REP_ESPACIAL_DECLARADA"].replace({
            "Não classificada" : "Nao declarado", 
            "Escala de Bairro": "Bairro",
            "Escala bairo": "Bairro",
            "Escala urbana": "Urbana",
            "Micro": "Microescala",
            "Media": "Mesoescala",
            "Média": "Mesoescala",
            "-": "Nao declarado",
            "Naoreconhecida": "Nao declarado"
        })

    if "FINALIDADE" in df.columns:
        df["FINALIDADE"] = df["FINALIDADE"].replace({
            "Licenciamento" : "Licenciamento Ambiental", 
            "Fornecer Dados": "Fornecer dados"
        })

    if "MONITORAR" in df.columns:
        df["FINALIDADE"] = df["FINALIDADE"].replace({
            "Não" : "Nao"
        })

    if "REALOCACAO" in df.columns:
        df["REALOCACAO"] = df["REALOCACAO"].replace({
            "Não" : "Nao"
        })

    if "MONITORAR" in df.columns:
        df["MONITORAR"] = df["MONITORAR"].replace({
            "Sm": "Sim"
        })

    if "MARCA" in df.columns:
        df["MARCA"] = df["MARCA"].replace({
            "x": "Nao declarado"
        })

    if "FONTE" in df.columns:
        df["FONTE"] = df["FONTE"].replace({
            "Consulta Interna": "Coleta interna",
            "Coleta Interna": "Coleta interna"
        })

    if "MOBILIDADE" in df.columns:
        df["MOBILIDADE"] = df["MOBILIDADE"].replace({
            "Móvel": "Movel"
        })

    if "CALIBRACAO" in df.columns:
        df["CALIBRACAO"] = df["CALIBRACAO"].replace({
            "diaria": "Diaria",
            "diária": "Diaria",
            "A cada 3 meses": "Trimestral",
            "A cada 1 mes": "Mensal",
            "A cada 1 mês": "Mensal",
            "Nao Aplicavel": "Nao aplicavel",
            "-": "Nao declarado",
            "x": "Nao declarado",
            "Não Aplicável": "Nao declarado",
            "A cada 2 meses": "Bimestral",
            "Mensal(O3),Mensal(NO2),Mensal(MP25)": "Mensal",
            "Sim": "Calibração feita - Sem especificacao"
            
        })

    # FALTA PADRONIZAR REPRESENTAÇÃO ESPACIAL E REPRESENTAÇÃO ESPACIAL DECLARADA
        
    return df

In [831]:
df_full = replace_vals(df_full)

In [832]:
# Consertar pequenas inconsistências 

mask = (df_full["UF"].isna() | df_full["UF"].astype(str).str.strip().eq("")) & \
       df_full["CIDADE"].astype(str).str.strip().str.casefold().eq("cobija")
df_full.loc[mask, "UF"] = "AC"

##### Completar com código dos municípios IBGE

In [834]:
def normalize_txt(s):
    s = "" if pd.isna(s) else str(s).strip()
    s = ud.normalize("NFKD", s)
    s = "".join(ch for ch in s if not ud.combining(ch))
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    return s

base = Path.cwd().parent
out_dir = base / "data" / "dicionarios"

# Try common Brazilian CSV settings
try:
    cd_mun = pd.read_csv(out_dir / "IBGE_CODIGO_MUN.csv", encoding="cp1252", sep=";")
except Exception:
    # Fallbacks
    try:
        cd_mun = pd.read_csv(out_dir / "IBGE_CODIGO_MUN.csv", encoding="latin1", sep=";")
    except Exception:
        cd_mun = pd.read_csv(out_dir / "IBGE_CODIGO_MUN.csv", encoding="cp1252")  # sep=','

In [838]:
def normalize_city(s):
    s = "" if pd.isna(s) else str(s).strip()
    s = re.sub(r"\(.*?\)", "", s)
    s = s.split(" - ")[0]
    s = ud.normalize("NFKD", s)
    s = "".join(ch for ch in s if not ud.combining(ch))
    s = re.sub(r"[^\w\s]", " ", s).lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _is_empty(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    return s.isna() | s.str.strip().isin(["", "nan", "none", "<NA>"])

# Build lookup
ibge_lu = cd_mun.rename(columns={
    "CÓDIGO DO MUNICÍPIO - IBGE": "CD_MUN_IBGE",
    "MUNICÍPIO - IBGE": "IBGE_CITY"
})[["UF", "IBGE_CITY", "CD_MUN_IBGE"]].copy()

# Normalize types and city names
ibge_lu["UF"] = ibge_lu["UF"].astype("string").str.strip().str.upper()
# ensure IBGE code is 7-digit STRING
ibge_lu["CD_MUN_IBGE"] = (
    pd.to_numeric(ibge_lu["CD_MUN_IBGE"], errors="coerce")
      .astype("Int64")
      .astype(str)
      .str.replace("<NA>", "", regex=False)
      .str.zfill(7)
      .astype("string")
)
ibge_lu["NORM_CITY"] = ibge_lu["IBGE_CITY"].apply(normalize_city)
ibge_lu = ibge_lu[["UF", "NORM_CITY", "CD_MUN_IBGE"]].drop_duplicates()

def fill_cd_mun_exact(df_full, uf_col="UF", city_col="CIDADE", code_col="CD_MUN"):
    out = df_full.copy()

    # normalize types
    out[uf_col]   = out[uf_col].astype("string").str.strip().str.upper()
    out[city_col] = out[city_col].astype("string")
    # make target column STRING too (so assignment matches)
    out[code_col] = out.get(code_col, pd.Series(index=out.index)).astype("string")

    out["_NORM_CITY"] = out[city_col].apply(normalize_city)

    m = out.merge(
        ibge_lu, left_on=[uf_col, "_NORM_CITY"],
        right_on=["UF", "NORM_CITY"], how="left"
    )

    empty = _is_empty(m[code_col])
    has_ibge = ~_is_empty(m["CD_MUN_IBGE"])

    # both are strings now → no TypeError
    m.loc[empty & has_ibge, code_col] = m.loc[empty & has_ibge, "CD_MUN_IBGE"]

    return (
        m.drop(columns=[c for c in ["UF_y", "NORM_CITY", "_NORM_CITY", "CD_MUN_IBGE"] if c in m.columns])
         .rename(columns={"UF_x": uf_col})
    )

# Usage
df_full = fill_cd_mun_exact(df_full.copy())

##### Criar ID_MMA e ID_MMA_COMPLETO para estações que não foram preenchidas ainda

In [839]:
# Investigar quantidade de estações nulas e não nulas para IDs
# total de linhas
total = len(df_full)
# quantas estão nulas
nulas = df_full["ID_MMA"].isna().sum()
# quantas não estão nulas
nao_nulas = total - nulas

print(f"Total: {total}")
print(f"Nulas: {nulas}")
print(f"Não nulas: {nao_nulas}")

Total: 2365
Nulas: 162
Não nulas: 2203


In [840]:
# Completar ID_MMA nas estações faltantes por estado

def _ascii_lower(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = ud.normalize("NFKD", s)
    s = "".join(ch for ch in s if not ud.combining(ch))
    return s.lower()

def _normalize_dt_mixed(series: pd.Series, anchor="start") -> pd.Series:
    s_raw = series.astype("string").str.strip().str.replace(r"[\/\.]", "-", regex=True)
    out = pd.to_datetime(s_raw, errors="coerce", utc=True)

    # YYYY-MM
    m_ym = s_raw.str.match(r"^\d{4}-\d{1,2}$", na=False)
    if m_ym.any():
        base = pd.to_datetime(s_raw[m_ym] + "-01", format="%Y-%m-%d", utc=True, errors="coerce")
        out.loc[m_ym] = base if anchor == "start" else (base + pd.offsets.MonthEnd(0))

    # YYYY
    m_y = s_raw.str.match(r"^\d{4}$", na=False)
    if m_y.any():
        suffix = "-01-01" if anchor == "start" else "-12-31"
        out.loc[m_y] = pd.to_datetime(s_raw[m_y] + suffix, format="%Y-%m-%d", utc=True, errors="coerce")

    return out.dt.tz_convert(None)

def _parse_existing_nums(id_series: pd.Series, uf: str) -> pd.Series:
    """Extrai os 4 dígitos finais dos IDs válidos daquela UF."""
    pat = rf"^{uf}\d{{4}}$"
    ok = id_series.astype("string").str.fullmatch(pat, na=False)
    return id_series.where(ok).str[-4:].astype("Int64", errors="ignore")

def _normalize_existing_ids_for_uf(df: pd.DataFrame, uf_col="UF", id_col="ID_MMA") -> pd.DataFrame:
    """Normaliza formatos como 'UF-7', 'UF 12', 'UF001' → 'UF0007', etc."""
    out = df.copy()
    if id_col not in out:
        out[id_col] = pd.NA
        return out
    uf = out[uf_col].astype("string").str.upper().fillna("")
    s  = out[id_col].astype("string")

    # se começa com UF, remove separadores até os dígitos
    m = s.str.match(r"^[A-Za-z]{2}\D*\d{1,4}$", na=False)
    tmp = s.where(m).str.replace(r"^([A-Za-z]{2})\D*(\d{1,4})$", lambda m: m.group(1)+m.group(2).zfill(4), regex=True)
    out.loc[m, id_col] = tmp
    out[id_col] = out[id_col].astype("string")
    return out

# --- principal ---

def fill_missing_id_mma(
    df: pd.DataFrame,
    uf_col="UF",
    start_col="INICIO",
    station_col="ID_OEMA",
    id_col="ID_MMA",
    anchor="start",
) -> pd.DataFrame:
    """
    Preenche IDs faltantes por UF, começando do maior ID existente.
    Ordenação: INICIO asc; empate/NaT por ID_OEMA A>Z.
    """
    out = df.copy()

    # tipos mínimos
    out[uf_col] = out.get(uf_col, pd.Series(pd.NA, index=out.index)).astype("string").str.upper()
    out[station_col] = out.get(station_col, pd.Series(pd.NA, index=out.index)).astype("string")
    if id_col not in out.columns:
        out[id_col] = pd.Series(pd.NA, index=out.index, dtype="string")
    else:
        out[id_col] = out[id_col].astype("string")

    # normaliza IDs existentes tipo UFdddd (opcional mas recomendado)
    out = _normalize_existing_ids_for_uf(out, uf_col=uf_col, id_col=id_col)

    # normaliza datas para ordenação
    out[start_col] = _normalize_dt_mixed(out[start_col], anchor=anchor)

    # chave de nome para desempate A>Z (usar lowercase ascii, mas invertendo a ordem depois)
    name_key = out[station_col].map(_ascii_lower)

    # ordena: UF, data asc, nome asc; depois vamos atribuir na ordem do grupo
    out = (
        out.assign(_name_key=name_key)
           .sort_values([uf_col, start_col, "_name_key"], kind="mergesort", na_position="last")
    )

    # preenche por UF
    filled = []
    for uf, g in out.groupby(uf_col, sort=False, dropna=False):
        gg = g.copy()

        # maior existente nessa UF
        existing_nums = _parse_existing_nums(gg[id_col], uf if isinstance(uf, str) else "")
        max_existing = int(existing_nums.max()) if not existing_nums.dropna().empty else 0

        # apenas onde está vazio
        to_fill = gg[id_col].isna() | gg[id_col].str.strip().eq("")
        idx = gg.index[to_fill]
        if len(idx) > 0:
            start_n = max_existing + 1
            seq = pd.Series(range(start_n, start_n + len(idx)), index=idx)
            new_ids = (str(uf) + seq.astype(int).astype(str).str.zfill(4)).astype("string")
            gg.loc[idx, id_col] = new_ids

        filled.append(gg)

    out2 = pd.concat(filled).sort_index()
    return out2.drop(columns=["_name_key"], errors="ignore")

In [843]:
df_final = fill_missing_id_mma(df_full.copy())

In [844]:
# Completar ID_MMA_COMPLETO nas estações faltantes por estado

_MISSING_TOKENS = {"", "na", "nan", "n/a", "-", "none", "null"}

def _normalize_missing(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    m = s.isna() | s.str.casefold().isin(_MISSING_TOKENS)
    return s.mask(m, pd.NA)

def _is_blank_series(s: pd.Series) -> pd.Series:
    return _normalize_missing(s).isna()

def _str_no_dotzero(s: pd.Series) -> pd.Series:
    s = _normalize_missing(s).fillna("")
    return s.str.replace(r"\.0$", "", regex=True)

def _cod_poluente_clean(s: pd.Series) -> pd.Series:
    s = _normalize_missing(s)
    s_num = pd.to_numeric(s, errors="coerce")
    out = _str_no_dotzero(s.fillna(""))
    m = s_num.notna()
    out.loc[m] = s_num.loc[m].astype("Int64").astype(str).str.zfill(3)
    # se veio "NA", "N/A" etc e não é número, vira vazio
    out = out.where(~_is_blank_series(out), "")
    return out.fillna("")

def _extract_uf_num(idmma: pd.Series) -> pd.DataFrame:
    s = _normalize_missing(idmma).fillna("")
    uf = s.str.extract(r"^([A-Z]{2})", expand=False)
    num = s.str.extract(r"^[A-Z]{2}\s*0*?(\d+)$", expand=False)
    num = pd.to_numeric(num, errors="coerce").astype("Int64")
    return pd.DataFrame({"_UF": uf, "_NUM": num}, index=s.index)

def _pair_for_row(uf: Optional[str], num: Optional[int]) -> str:
    if uf is None or pd.isna(uf):
        return ""
    uf = str(uf).upper()

    fixed_default = {
        "SP": "RA", "ES": "RA", "MG": "RA", "SC": "RA", "RS": "RA",
        "PR": "RA", "BA": "ND", "MA": "RA", "MT": "IA", "PE": "ND",
        "RR": "RS", "PB": "ND", "CE": "ND",
    }

    def ge(n: Optional[int], cutoff: int) -> bool:
        return n is not None and not pd.isna(n) and int(n) >= cutoff

    if uf == "RJ":
        return "ND" if ge(num, 1000) else "RA"
    if uf == "PA":
        return "ND" if ge(num, 1001) else "RA"
    if uf == "RN":
        return "ND" if ge(num, 1001) else "RA"

    return fixed_default.get(uf, "RA")

def make_id_mma_completo_fixos(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    idmma = _str_no_dotzero(out.get("ID_MMA", pd.Series(index=out.index)))
    cod   = _cod_poluente_clean(out.get("COD_POLUENTE", ""))

    meta = _extract_uf_num(idmma)
    uf, num = meta["_UF"], meta["_NUM"]

    pair = pd.Series((_pair_for_row(u, int(n) if not pd.isna(n) else None)
                      for u, n in zip(uf, num)), index=out.index, dtype="string")

    out["ID_MMA_COMPLETO"] = (idmma.fillna("") + pair.fillna("") + cod).astype("string")
    # se toda a composição ficou vazia, mantém vazio
    out.loc[_is_blank_series(idmma) & _is_blank_series(cod), "ID_MMA_COMPLETO"] = ""
    return out

# ----------------- preencher só vazios de ID_MMA_COMPLETO -----------------
def complete_empy_MMA_COMPLETO(df_final: pd.DataFrame) -> pd.DataFrame:
    df = df_final.copy()

    # normaliza colunas usadas
    if "ID_MMA" in df.columns:
        df["ID_MMA"] = _normalize_missing(df["ID_MMA"]).fillna("")
    if "COD_POLUENTE" in df.columns:
        df["COD_POLUENTE"] = _normalize_missing(df["COD_POLUENTE"]).fillna("")

    # garante coluna de destino
    if "ID_MMA_COMPLETO" not in df.columns:
        df["ID_MMA_COMPLETO"] = pd.Series(index=df.index, dtype="string")

    calc = make_id_mma_completo_fixos(df)
    m_vazio_destino = _is_blank_series(df["ID_MMA_COMPLETO"])

    df.loc[m_vazio_destino, "ID_MMA_COMPLETO"] = calc.loc[m_vazio_destino, "ID_MMA_COMPLETO"]

    # opcional: normaliza destino para não deixar "NA" etc
    df["ID_MMA_COMPLETO"] = _normalize_missing(df["ID_MMA_COMPLETO"]).fillna("")

    return df

In [847]:
df_final = complete_empy_MMA_COMPLETO(df_final.copy())
df_final

,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,ANOS_MONITORADOS,BASE_DADOS,ELEVACAO,REALOCACAO,OBS_CALIBRACAO,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,REP_ESPACIAL_DECLARADA,OPERACAO
0,AC,Cobija,<NA>,Plaza Principal Cobija,AC0030,AC0030RA002,<NA>,Publica,<NA>,<NA>,...,"2022,2023,2024,2025",NaN,675.0,NaN,NaN,NaN,Nao reconhecida,NaN,NaN,NaN
1,AC,Rio Branco,1200401,AcreBioClima - UFAC,AC0001,AC0001RA002,<NA>,Publica,<NA>,<NA>,...,"2019,2020,2021,2022,2023,2024,2025",NaN,524.0,NaN,NaN,NaN,Reconhecida,<NA>,<NA>,<NA>
2,AC,Rio Branco,1200401,Ministério Público do Estado do Acre (SEDE),AC0002,AC0002RA002,<NA>,Publica,<NA>,<NA>,...,"2019,2020,2021,2022,2023,2024,2025",NaN,495.0,NaN,NaN,NaN,Reconhecida,<NA>,<NA>,<NA>
3,AC,Assis Brasil,1200054,MPAC_ABR_01_promotoria,AC0003,AC0003RA002,<NA>,Publica,<NA>,<NA>,...,"2019,2020,2021,2022,2023,2024,2025",NaN,783.0,NaN,NaN,NaN,Reconhecida,<NA>,<NA>,<NA>
4,AC,Assis Brasil,1200054,MPAC_ABR_02_SEMSA,AC0004,AC0004RA002,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2360,SP,Taubaté,3554102.0,Taubaté,SP0280,SP0280RA003,CETESB,Publica,CETESB,Pública,...,NaN,NaN,NaN,Nao,<NA>,<NA>,,<NA>,Bairro,<NA>
2361,SP,São Paulo,3550308,Vielas da Água Preta,SP0323,SP0323RA002,<NA>,Publica,<NA>,<NA>,...,2025,NaN,2435.0,NaN,NaN,NaN,Nao reconhecida,NaN,NaN,NaN
2362,TO,<NA>,<NA>,(IPAM) TI Xerente,TO0003,TO0003RA002,<NA>,Publica,<NA>,<NA>,...,"2024,2025",NaN,1068.0,NaN,NaN,NaN,Nao reconhecida,NaN,NaN,NaN
2363,TO,Palmas,1721000,MPTO_PMW_01_procuradoriageral,TO0001,TO0001RA002,<NA>,Publica,<NA>,<NA>,...,"2020,2021,2022,2023,2024,2025",NaN,850.0,NaN,NaN,NaN,Nao reconhecida,NaN,NaN,NaN


In [374]:
# def _ascii_lower(s: str) -> str:
#     s = "" if pd.isna(s) else str(s)
#     s = ud.normalize("NFKD", s)
#     s = "".join(ch for ch in s if not ud.combining(ch))
#     return s.lower()

# def _normalize_dt_utc_naive(series: pd.Series) -> pd.Series:
#     # Parse everything with utc=True to handle mixed tz, then strip tz info
#     s = pd.to_datetime(series, errors="coerce", utc=True)
#     return s.dt.tz_convert(None)

# def assign_id_mma(df, uf_col="UF", start_col="INICIO", name_col="ID_OEMA", only_missing=False):
#     out = df.copy()

#     # types
#     out[uf_col]   = out[uf_col].astype("string")
#     out[name_col] = out[name_col].astype("string")

#     # normalize datetime: make all UTC then remove tz
#     out[start_col] = _normalize_dt_utc_naive(out[start_col])

#     # stable sort by UF, INICIO, name
#     name_key = out[name_col].map(_ascii_lower)
#     out = (
#         out.assign(_name_key=name_key)
#            .sort_values([uf_col, start_col, "_name_key"], kind="mergesort", na_position="last")
#            .drop(columns="_name_key")
#            .reset_index(drop=True)
#     )

#     # sequence per UF
#     mask_valid_uf = out[uf_col].notna() & out[uf_col].str.strip().ne("")
#     seq = (
#         out.loc[mask_valid_uf]
#            .groupby(uf_col)
#            .cumcount()
#            .add(1)
#            .astype(str)
#            .str.zfill(4)
#     )
#     new_ids = (out.loc[mask_valid_uf, uf_col] + seq).astype("string")

#     if only_missing:
#         mask_missing_id = out.get("ID_MMA", pd.Series(index=out.index)).astype("string").str.strip().isin([None, "", "nan"])
#         to_set = mask_valid_uf & mask_missing_id
#         out.loc[to_set, "ID_MMA"] = new_ids
#     else:
#         out["ID_MMA"] = out.get("ID_MMA", pd.Series(index=out.index, dtype="string"))
#         out.loc[mask_valid_uf, "ID_MMA"] = new_ids

#     out["ID_MMA"] = out["ID_MMA"].astype("string")
#     return out

In [522]:
# base = Path.cwd().parent  # .../RQAR_2025_book
# out_dir = base / "data" 
# out_dir.mkdir(parents=True, exist_ok=True)

# out_file = out_dir / "testeRafa.csv"
# teste.to_csv(out_file, index=False, encoding="utf-8")
# print("Saved to:", out_file.resolve())

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/testeRafa.csv


In [848]:
def fill_declared_all_cols(df):
    out = df.copy().astype("string")
    out = out.replace(r"^\s*$", np.nan, regex=True)
    out = out.replace({"nan": np.nan, "NaN": np.nan, "none": np.nan, "None": np.nan})
    out = out.fillna("Nao declarado")  # fills NaN, <NA>, NaT
    return out

df_final = fill_declared_all_cols(df_final.copy())

In [448]:
# # Criar ID_MMA_COMPLETO

# def create_ID_MMA_COMPLETO(df):
#     df["ID_MMA_COMPLETO"] = (
#         df["ID_MMA"].astype(str)
#         + df.get("CATEGORIA", "").astype(str).str[:1].fillna("")
#         + df.get("FUNCIONAMENTO", "").astype(str).str[:1].fillna("")
#         + df["COD_POLUENTE"]
#     )
    
#     df_final.append(df)
    
#     # --- Concatenar todos os dados ---
#     df_final = pd.concat(df_final, ignore_index=True)

In [458]:
# def _str_no_dotzero(s: pd.Series) -> pd.Series:
#     # string, trim e remove sufixo ".0"
#     return s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

# def _first_letter_keep(s: pd.Series) -> pd.Series:
#     # primeira letra/dígito da string (ignora espaços/pontuação)
#     t = s.astype("string").fillna("").str.strip()
#     first = t.str.extract(r"([A-Za-z0-9])", expand=False)
#     return first.fillna("").str.upper()

# def _cod_poluente_clean(s: pd.Series) -> pd.Series:
#     # se for numérico → zfill(3); senão, remove ".0" e espaços
#     s = s.astype("string")
#     s_num = pd.to_numeric(s, errors="coerce")
#     out = _str_no_dotzero(s)
#     m = s_num.notna()
#     out.loc[m] = s_num.loc[m].astype("Int64").astype(str).str.zfill(3)
#     return out.fillna("")

# def make_id_mma_completo(df: pd.DataFrame) -> pd.DataFrame:
#     out   = df.copy()
#     idmmA = _str_no_dotzero(out.get("ID_MMA", pd.Series(index=out.index)))
#     cat   = _first_letter_keep(out.get("CATEGORIA", ""))       # pega N de "Não declarado"
#     func  = _first_letter_keep(out.get("FUNCIONAMENTO", ""))   # idem
#     cod   = _cod_poluente_clean(out.get("COD_POLUENTE", ""))

#     out["ID_MMA_COMPLETO"] = (idmmA.fillna("") + cat + func + cod).astype("string")
#     return out

In [464]:
# teste = make_id_mma_completo(teste)

In [849]:
df_final.groupby('POLUENTE').count()

,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,ANOS_MONITORADOS,BASE_DADOS,ELEVACAO,REALOCACAO,OBS_CALIBRACAO,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,REP_ESPACIAL_DECLARADA,OPERACAO
POLUENTE,,,,,,,,,,,,,,,,,,,,,
ACETAL,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
BENZENO,39,39,39,39,39,39,39,39,39,39,...,39,39,39,39,39,39,39,39,39,39
CH4,65,65,65,65,65,65,65,65,65,65,...,65,65,65,65,65,65,65,65,65,65
CO,192,192,192,192,192,192,192,192,192,192,...,192,192,192,192,192,192,192,192,192,192
ERT,3,3,3,3,3,3,3,3,3,3,...,3,3,3,3,3,3,3,3,3,3
ETILBENZENO,26,26,26,26,26,26,26,26,26,26,...,26,26,26,26,26,26,26,26,26,26
FMC,7,7,7,7,7,7,7,7,7,7,...,7,7,7,7,7,7,7,7,7,7
FORMAL,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
H2S,10,10,10,10,10,10,10,10,10,10,...,10,10,10,10,10,10,10,10,10,10


9. Salvar e exportar

In [850]:
base = Path.cwd().parent  # .../RQAR_2025_book
out_dir = base / "data" 
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "TESTE.csv" #Monitoramento_QAr_BR
df_final.to_csv(out_file, index=False, encoding="utf-8")
print("Saved to:", out_file.resolve())

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/TESTE.csv


##### Correções depois do arquivo salvo 

In [9]:
base = Path.cwd().parent
df_dir = base / "data"
file_path = df_dir / "Monitoramento_QAr_BR.csv"

base = pd.read_csv(file_path, sep=",", encoding="utf-8")
base

,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,ANOS_MONITORADOS,BASE_DADOS,ELEVACAO,REALOCACAO,OBS_CALIBRACAO,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,REP_ESPACIAL_DECLARADA,OPERACAO
0,AC,Cobija,Nao declarado,Plaza Principal Cobija,AC0030,AC0030RA002,Nao declarado,Publica,Nao declarado,Nao declarado,...,"2022,2023,2024,2025",Nao declarado,675.0,Nao declarado,Nao declarado,Nao declarado,Nao reconhecida,Nao declarado,Nao declarado,Nao declarado
1,AC,Rio Branco,1200401,AcreBioClima - UFAC,AC0001,AC0001RA002,Nao declarado,Publica,Nao declarado,Nao declarado,...,"2019,2020,2021,2022,2023,2024,2025",Nao declarado,524.0,Nao declarado,Nao declarado,Nao declarado,Reconhecida,Nao declarado,Nao declarado,Nao declarado
2,AC,Rio Branco,1200401,Ministério Público do Estado do Acre (SEDE),AC0002,AC0002RA002,Nao declarado,Publica,Nao declarado,Nao declarado,...,"2019,2020,2021,2022,2023,2024,2025",Nao declarado,495.0,Nao declarado,Nao declarado,Nao declarado,Reconhecida,Nao declarado,Nao declarado,Nao declarado
3,AC,Assis Brasil,1200054,MPAC_ABR_01_promotoria,AC0003,AC0003RA002,Nao declarado,Publica,Nao declarado,Nao declarado,...,"2019,2020,2021,2022,2023,2024,2025",Nao declarado,783.0,Nao declarado,Nao declarado,Nao declarado,Reconhecida,Nao declarado,Nao declarado,Nao declarado
4,AC,Assis Brasil,1200054,MPAC_ABR_02_SEMSA,AC0004,AC0004RA002,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2361,SP,São Paulo,3550308,Vielas da Água Preta,SP0323,SP0323RA002,Nao declarado,Publica,Nao declarado,Nao declarado,...,2025,Nao declarado,2435.0,Nao declarado,Nao declarado,Nao declarado,Nao reconhecida,Nao declarado,Nao declarado,Nao declarado
2362,TO,Nao declarado,Nao declarado,(IPAM) TI Xerente,TO0003,TO0003RA002,Nao declarado,Publica,Nao declarado,Nao declarado,...,"2024,2025",Nao declarado,1068.0,Nao declarado,Nao declarado,Nao declarado,Nao reconhecida,Nao declarado,Nao declarado,Nao declarado
2363,TO,Palmas,1721000,MPTO_PMW_01_procuradoriageral,TO0001,TO0001RA002,Nao declarado,Publica,Nao declarado,Nao declarado,...,"2020,2021,2022,2023,2024,2025",Nao declarado,850.0,Nao declarado,Nao declarado,Nao declarado,Nao reconhecida,Nao declarado,Nao declarado,Nao declarado
2364,TO,Nao declarado,Nao declarado,REP-Colinas do Tocantins,TO0002,TO0002RA002,Nao declarado,Publica,Nao declarado,Nao declarado,...,"2021,2022,2023,2024,2025",Nao declarado,739.0,Nao declarado,Nao declarado,Nao declarado,Nao reconhecida,Nao declarado,Nao declarado,Nao declarado


In [864]:
base.isna().sum().sum()

np.int64(40)

In [863]:
for col in base.columns:
    s = base[col]
    vc = s.dropna().value_counts()
    print(vc) 

UF
RJ    742
SP    434
MG    355
ES    124
PR     99
BA     85
RS     84
AM     64
CE     52
MT     49
AL     46
MA     42
AC     31
MS     24
PA     23
SC     22
DF     22
PE     21
RR     19
PB     12
AP      6
RO      6
TO      3
Name: count, dtype: int64
CIDADE
Rio de Janeiro    235
Nao declarado     167
São Paulo         129
Itaborai           62
Macae              62
                 ... 
Cordeirópolis       1
Guarujá             1
Jaboticabal         1
Itu                 1
Palmas              1
Name: count, Length: 210, dtype: int64
CD_MUN
3304557.0        234
Nao declarado    185
3550308.0        121
3302403.0         62
3301702.0         62
                ... 
3513504            1
1400100            1
3523909.0          1
3524303.0          1
1721000            1
Name: count, Length: 207, dtype: int64
ID_OEMA
VR - Santa Cecilia                 16
VR - Retiro                        16
VR - Belmonte                      16
RJ - Adalgisa Nery                 16
RJ - Ilha de Paq

##### Rascunhos

In [122]:
# from rapidfuzz import fuzz, process

# # ---------------- Params ----------------
# id_col = "ID_OEMA"
# uf_col = "UF"
# lat_col = "LATITUDE"
# lon_col = "LONGITUDE"
# min_score = 70  # raise to be stricter
# max_km = 0.1      # set None to skip distance filtering
# topn = 3          # top candidates per left item

# # -------------- Helpers -----------------
# def norm_str(s: pd.Series) -> pd.Series:
#     return s.astype("string").fillna("").str.strip().str.upper()

# def to_float(s: pd.Series) -> pd.Series:
#     return pd.to_numeric(s.astype(str).str.replace(",", "."), errors="coerce")

# def haversine_km(lat1, lon1, lat2, lon2):
#     R = 6371.0088
#     lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
#     dlat = lat2 - lat1
#     dlon = lon2 - lon1
#     a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
#     return 2 * R * np.arcsin(np.sqrt(a))

# # ------------- Normalize ----------------
# for df in (df_mma, df_purple):
#     df["_uf"]  = norm_str(df[uf_col])
#     df["_id"]  = norm_str(df[id_col])  # use ID text as the comparable string
#     df["_lat"] = to_float(df.get(lat_col, pd.Series(index=df.index, dtype="float64")))
#     df["_lon"] = to_float(df.get(lon_col, pd.Series(index=df.index, dtype="float64")))

# # keep original indices to map rows back after merge
# mma_i = df_mma.reset_index().rename(columns={"index": "idx_mma"})
# pur_i = df_purple.reset_index().rename(columns={"index": "idx_purple"})

# # ---------- Exact matches UF + ID -------
# exact = mma_i.merge(
#     pur_i,
#     on=["_uf", "_id"],
#     how="inner",
#     suffixes=("_mma", "_purple")
# )
# print("Exact matches UF+ID:", len(exact))

# # indices matched on each side
# mma_matched_idx = set(exact["idx_mma"])
# pur_matched_idx = set(exact["idx_purple"])

# # ---------- Unmatched for fuzzy ----------
# mma_unmatched = df_mma.loc[~df_mma.index.isin(mma_matched_idx), ["_uf","_id","_lat","_lon", id_col]]
# pur_unmatched = df_purple.loc[~df_purple.index.isin(pur_matched_idx), ["_uf","_id","_lat","_lon", id_col]]

# # ---------- Fuzzy within same UF --------
# def candidates_same_uf(left: pd.DataFrame, right: pd.DataFrame,
#                        min_score=90, max_km=1.0, topn=3) -> pd.DataFrame:
#     rows = []
#     for uf, lgrp in left.groupby("_uf"):
#         rgrp = right[right["_uf"] == uf]
#         if rgrp.empty:
#             continue

#         choices = rgrp["_id"].dropna().unique().tolist()
#         if not choices:
#             continue

#         for li, lid in lgrp["_id"].dropna().items():
#             hits = process.extract(lid, choices, scorer=fuzz.token_set_ratio,
#                                    limit=topn, score_cutoff=min_score)
#             for rid_text, score, _ in hits:
#                 # link back to concrete rows for this UF
#                 for lidx in lgrp.index[lgrp["_id"] == lid]:
#                     for ridx in rgrp.index[rgrp["_id"] == rid_text]:
#                         rows.append({"UF": uf, "li": lidx, "ri": ridx, "score": score})

#     out = pd.DataFrame(rows)
#     if out.empty:
#         return out

#     # attach original values
#     out = (
#         out.merge(
#             df_mma[[id_col, lat_col, lon_col, "_lat", "_lon"]],
#             left_on="li", right_index=True
#         ).merge(
#             df_purple[[id_col, lat_col, lon_col, "_lat", "_lon"]],
#             left_on="ri", right_index=True, suffixes=("_mma", "_purple")
#         )
#     )

#     # optional distance filter if coords are present
#     if max_km is not None:
#         has_coords = out[["_lat_mma","_lon_mma","_lat_purple","_lon_purple"]].notna().all(axis=1)
#         if has_coords.any():
#             out.loc[has_coords, "dist_km"] = haversine_km(
#                 out.loc[has_coords, "_lat_mma"],
#                 out.loc[has_coords, "_lon_mma"],
#                 out.loc[has_coords, "_lat_purple"],
#                 out.loc[has_coords, "_lon_purple"],
#             )
#             out = out[(out["dist_km"] <= max_km) | out["dist_km"].isna()]
#         else:
#             out["dist_km"] = np.nan
#     else:
#         out["dist_km"] = np.nan

#     # remove trivial pairs if the textual IDs are exactly equal
#     out = out[out[f"{id_col}_mma"].astype("string") != out[f"{id_col}_purple"].astype("string")]
    
#     return (
#         out.rename(columns={"_uf":"UF"})
#           [[
#               "UF",
#               f"{id_col}_mma", f"{id_col}_purple",
#               # cleaned numeric coords
#               "_lat_mma", "_lon_mma", "_lat_purple", "_lon_purple",
#               # raw text coords if you want to inspect the originals
#               f"{lat_col}_mma", f"{lon_col}_mma", f"{lat_col}_purple", f"{lon_col}_purple",
#               "score", "dist_km"
#           ]]
#           .drop_duplicates()
#           .sort_values(["score","dist_km"], ascending=[False, True])
#           .reset_index(drop=True)
#     )

# likely_same_diff_id = candidates_same_uf(
#     mma_unmatched, pur_unmatched,
#     min_score=min_score, max_km=max_km, topn=topn
# )
# likely_same_diff_id 

In [196]:
# base = Path.cwd().parent  
# out_dir = base / "data" / "DADOS_ESTACOES" / "Indicativas" 
# out_dir.mkdir(parents=True, exist_ok=True)

# suspects_csv = out_dir / "FULL_suspects_same_station_diff_ids.csv"
# likely_same_diff_id.to_csv(suspects_csv, index=False, encoding="utf-8-sig")

In [123]:
# import unicodedata, re
# import pandas as pd

# to_filter = [
#     'AcreBioClima - UFAC A',
#     'MPAC_ABR_01_promotoria A',
#     'MPAC_BJR_01_promotoria A',
#     'MPAC_BRL_01_promotoria A',
#     'MPAC_FIJ_01_promotoria A',
#     'MPAC_JRD_01_prefeitura A',
#     'MPAC_PTA_01_Sec.infraestrutura A',
#     'MPAC_SNG_01_promotoria A',
#     'MPAC_SNM_01_ifac A',
#     'MPAC_TRC_02_ifac A',
#     'RB-BACKUP (Estacao particular FB) A',
#     'UFACFloresta A',
#     'Ministerio Publico do Estado do Acre (SEDE) A',
#     'MPAC_RDA_01_prefeitura A',
#     'MPAC_ACL_01_promotoria A',
#     'MPAC_MNU_01_promotoria A',
#     'MPAC_PLC_01_promotoria A',
#     'MPAC_SNM_02_promotoria A',
#     'MPAC_SRP_01_prefeitura A',
#     'MPAC_PTW_01_prefeitura A',
#     'MPAC_XAP_02_promotoria A',
#     'UFAC A'
# ]

# def fold(s: str) -> str:
#     if s is None: return ""
#     s = str(s).replace("\xa0", " ")
#     # remove accents
#     s = unicodedata.normalize("NFD", s)
#     s = "".join(ch for ch in s if not unicodedata.combining(ch))
#     s = unicodedata.normalize("NFKC", s)
#     # collapse spaces and upper
#     s = re.sub(r"\s+", " ", s).strip().upper()
#     return s

# # normalized targets
# target = {fold(x) for x in to_filter}

# # column as string and its folded version
# col = df_mma["ID_OEMA"].astype("string")
# col_fold = col.map(fold)

# # match only those rows present in your list (accent-insensitive)
# mask = col_fold.isin(target)

# # remove trailing "A" (with optional spaces) on matched rows only
# df_mma.loc[mask, "ID_OEMA"] = (
#     col.loc[mask]
#        .str.replace("\xa0", " ")
#        .str.replace(r"\s+", " ", regex=True)
#        .str.replace(r"\s*A\s*$", "", regex=True)
#        .str.strip()
# )
# df_mma

In [124]:
# import unicodedata
# import re

# to_filter = [
#     'AcreBioClima - UFAC A',
#     'MPAC_ABR_01_promotoria A',
#     'MPAC_BJR_01_promotoria A',
#     'MPAC_BRL_01_promotoria A',
#     'MPAC_FIJ_01_promotoria A',
#     'MPAC_JRD_01_prefeitura A',
#     'MPAC_PTA_01_Sec.infraestrutura A',
#     'MPAC_SNG_01_promotoria A',
#     'MPAC_SNM_01_ifac A',
#     'MPAC_TRC_02_ifac A',
#     'RB-BACKUP (Estacao particular FB) A',
#     'UFACFloresta A',
#     'Ministerio Publico do Estado do Acre (SEDE) A',
#     'MPAC_RDA_01_prefeitura A',
#     'MPAC_ACL_01_promotoria A',
#     'MPAC_MNU_01_promotoria A',
#     'MPAC_PLC_01_promotoria A',
#     'MPAC_SNM_02_promotoria A',
#     'MPAC_SRP_01_prefeitura A',
#     'MPAC_PTW_01_prefeitura A',
#     'MPAC_XAP_02_promotoria A',
#     'UFAC A'
# ]

# def norm_txt(s: str) -> str:
#     s = unicodedata.normalize("NFKC", str(s)).replace("\xa0", " ")
#     s = re.sub(r"\s+", " ", s).strip()
#     return s.upper()

# # normalized set of targets
# to_filter_norm = {norm_txt(x) for x in to_filter}

# # current column as string
# col = df_mma["ID_OEMA"].astype("string")

# # normalized view of the column
# col_norm = col.map(norm_txt)

# # mask: rows that match any item from the list (after normalization)
# mask = col_norm.isin(to_filter_norm)

# # remove trailing "A" and spaces only on the matched rows
# df_mma.loc[mask, "ID_OEMA"] = (
#     col.loc[mask]
#        .str.replace("\xa0", " ")            # replace non-breaking spaces
#        .str.replace(r"\s+", " ", regex=True)
#        .str.replace(r"\s*A\s*$", "", regex=True)  # drop trailing A
#        .str.strip()
# )

# print("Rows updated:", mask.sum())

# # Optional: show which list items were not found (debug)
# remaining = to_filter_norm.difference(set(col_norm[mask].unique()))
# if remaining:
#     print("Not found (normalized):", remaining)